# Asg2 – Step 2：實驗與消融（Q2 / Q3 / Q4）

這份 notebook 用**同一份 Dataset、同一個 M1 程式庫**（`asg2_lib.py`）跑一系列「有名字的設定」，每個設定的結果
**一跑完就寫進 `/kaggle/working/results.csv`**；超過時間或中斷後，下一個 session 會自動跳過已完成的設定繼續跑。

**評估流程（所有設定都一樣，全部依時間切分）**
* val 週 = 資料內最後一個完整週（與 t 同為 UTC 週六 16:00 開始）；calib 週 = 前一週。
* 訓練 cutoff 的標籤窗都在 calib 週的特徵截止時間（cutoff − gap）之前結束。
* epoch 只用 **calib 週的平均 AP** 挑（完全不看 val 週）；決策比例也只在 calib 週調 → val 週的 Score 是誠實的估計。
* 每個結果表都附 **All zeros、Copy last week、Copy last week (gap)**。

**哪些是講義方法、哪些是我們自己的設計**（每個自己的設計都有對應的消融）

| 項目 | 來源 | 消融設定 |
|---|---|---|
| RDL 異質圖 GNN（M1，RelBench GraphSAGE） | 講義 L5-1 p22–p29；L5-2 p20 | `M1_s42/43/44` |
| RelGNN atomic routes（M2，官方程式碼） | 講義 L5-2 p6–p9；arXiv:2502.06784 | `M2_RelGNN_s42/43/44` |
| 時間感知取樣、相對時間編碼 | L5-1 p29、p31、p35；L5-2 p11–p12 | `Q3_no_time_enc`、`Q3_uniform_sampling`、`Q3_lookback_*` |
| late-known 欄位只在 < t 時使用 | 規格 §2.4；L5-1 p10 | `Q3_late_dynamic` |
| 表 / 邊 / 層數 / 聚合 / 注意力 | L5-1 p15–p19、p30–p34；L5-2 p4 | `Q2_*` |
| 手工 COUNT 特徵 + LightGBM（tabular baseline） | L5-1 p10、p30；L5-2 p8 | `LightGBM_counts` |
| **決策規則：依比例取前 r%（我們的設計）** | 自己的設計 | 每一列都同時記錄 `score_calib_probthr`、`score_th05` |
| **cohort 活動量情境 ctx（我們的設計）** | 自己的設計 | `Q3_no_global_ctx` |
| **台灣時間 hour / 星期 sin-cos（我們的設計）** | 自己的設計（參考 L5-1 p35 seasonal features） | `Q3_no_cyclic` |
| **gap 模擬（我們的設計）** | 自己的設計 | `Q3_no_gap_sim` |

**執行前**：Accelerator = **GPU T4 x2**、Internet = **On**、Add Input = 你的 Dataset（`asg2-gharchive-db_v2`）。
完整清單在 GPU 上很可能超過一個 session（12 小時）→ 見最後一節「分多個 session 續跑」。

Data Source: GH Archive https://www.gharchive.org/

In [ ]:
# ===== 設定（只需要改這裡）=====
SMOKE = False              # True = 每個設定只跑幾步，確認程式能跑（數字沒有意義、不能放投影片）
RUN_LIST = "ALL"           # "ALL" = 完整清單（依優先順序）；或指定名稱，例如 ["M1_s42", "M2_RelGNN_s42", "Q3_no_time_enc"]
SESSION_BUDGET_H = 11.0    # 這個 session 最多跑幾小時（Kaggle 上限 12 小時；預估下一個設定會超過就先停，結果不會遺失）
PER_CONFIG_MAX_MIN = 600   # 單一設定的安全上限（分鐘）。設很大 = 每個設定都跑完整 epochs（8）＋early stopping，
                           # 只是保證一個設定能在一個 session 內跑完；算力不是問題，所以不為了時間縮短訓練
CFG_OVERRIDE = {}          # 進階：覆蓋 asg2_lib.Config（所有設定共用），例如 {"epochs": 8}
SEED_BASE = 42

DATA_ROOT  = "/kaggle/input"
WORK_DIR   = "/kaggle/working"
REPO_URL   = "https://github.com/nycubigdata-2026fall/BigData2026-Assignment2"
REPO_DIR   = "/kaggle/working/BigData2026-Assignment2"

import os, json
DATA_ROOT = os.environ.get("ASG2_DATA_ROOT", DATA_ROOT)
WORK_DIR  = os.environ.get("ASG2_WORK_DIR", WORK_DIR)
REPO_DIR  = os.environ.get("ASG2_REPO_DIR", REPO_DIR)
SMOKE     = os.environ.get("ASG2_SMOKE", str(SMOKE)) in ("True", "1", "true")
if os.environ.get("ASG2_RUN_LIST"): RUN_LIST = json.loads(os.environ["ASG2_RUN_LIST"])
CFG_OVERRIDE.update(json.loads(os.environ.get("ASG2_CFG_JSON", "{}")))
os.makedirs(WORK_DIR, exist_ok=True)
RESULTS = os.path.join(WORK_DIR, "results_smoke.csv" if SMOKE else "results.csv")
print("SMOKE =", SMOKE, "| RUN_LIST =", RUN_LIST, "| results ->", RESULTS)

## 1. 安裝套件（與 notebook 01 相同）

In [ ]:
import sys, subprocess, importlib.util, torch
print("python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA", torch.version.cuda,
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
def _have(m): return importlib.util.find_spec(m) is not None
def _pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    if r.returncode: print(r.stdout[-2000:], r.stderr[-3000:])
    return r.returncode == 0
open("/tmp/torch_constraint.txt", "w").write(f"torch=={torch.__version__}\n")
C = ["-c", "/tmp/torch_constraint.txt"]
if not all(_have(m) for m in ["torch_geometric", "torch_frame", "relbench"]):
    _pip("torch_geometric==2.8.1", "pytorch_frame==0.3.0", "relbench==3.1.0", *C)
def _pyg_candidates():
    # 依 torch / CUDA 版本列出 data.pyg.org 的 wheel 頁面（先完全相符，再同 CUDA 主版本最接近的）
    import re, urllib.request
    base = torch.__version__.split("+")[0]; mm0 = ".".join(base.split(".")[:2]) + ".0"
    cu = torch.version.cuda
    tag = "cu" + cu.replace(".", "") if cu else "cpu"
    try:
        idx = urllib.request.urlopen("https://data.pyg.org/whl/", timeout=30).read().decode()
        pages = set(re.findall(r"torch-([0-9.]+)\+(cu[0-9]+|cpu)", idx))
    except Exception as e:
        print("cannot read https://data.pyg.org/whl/ :", e); pages = {(base, tag), (mm0, tag)}
    out = []
    for v in [base, mm0]:
        if (v, tag) in pages: out.append((v, tag))
        if cu:   # 同 CUDA 主版本（例如 12.x）的其他 build，差距小的優先
            alts = [t for (vv, t) in pages if vv == v and t.startswith("cu") and t[2:4] == tag[2:4] and t != tag]
            out += [(v, t) for t in sorted(alts, key=lambda t: abs(int(t[2:]) - int(tag[2:])))]
    seen = []
    for c in out:
        if c not in seen: seen.append(c)
    return [f"https://data.pyg.org/whl/torch-{v}+{t}.html" for v, t in seen]

def _sampler_ok():
    r = subprocess.run([sys.executable, "-c", "import torch_geometric.typing as T; "
                        "assert T.WITH_PYG_LIB or T.WITH_TORCH_SPARSE; print('ok')"], capture_output=True, text=True)
    return "ok" in r.stdout

if not _sampler_ok():
    cands = _pyg_candidates(); print("pyg-lib wheel pages to try:", cands)
    ok = False
    for url in cands:
        for pkg in ["pyg-lib", "torch-sparse"]:
            if _pip(pkg, "--only-binary", ":all:", "-f", url, *C) and _sampler_ok():
                print("installed", pkg, "from", url); ok = True; break
            subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", pkg], capture_output=True)
        if ok: break
    if not ok:
        print("!!! 找不到對應 torch", torch.__version__, "CUDA", torch.version.cuda, "的 pyg-lib / torch-sparse wheel。")
        print("    請到 https://data.pyg.org/whl/ 找有 wheel 的 torch 版本，在這格最上面加")
        print("    !pip install torch==<那個版本> 之後 Run → Restart & Run All。詳見 TRAIN_GUIDE_zh.md。")
import torch_geometric, torch_frame, relbench
print("torch_geometric", torch_geometric.__version__, "| torch_frame", torch_frame.__version__, "| relbench", relbench.__version__)
try:
    import pyg_lib; print("pyg_lib", pyg_lib.__version__)
except ImportError:
    import torch_sparse; print("torch_sparse", torch_sparse.__version__)

## 2. 官方 repo（只讀 `test_input.csv`）

In [ ]:
if not os.path.exists(os.path.join(REPO_DIR, "scripts", "evaluate.py")):
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
TEST_INPUT = os.path.join(REPO_DIR, "submission", "test_input.csv")
EVALUATE   = os.path.join(REPO_DIR, "scripts", "evaluate.py")
print("official repo commit:", REPO_COMMIT or "(not a git checkout)")

## 3. 程式庫（不需要修改）
`asg2_lib.py` = notebook 01 的同一份；`asg2_exp.py` = 實驗設定與執行器；`relgnn_vendor.py` = RelGNN 官方程式碼（MIT）。

In [ ]:
%%writefile asg2_lib.py
# -*- coding: utf-8 -*-
"""asg2_lib.py — Assignment 2（預測 GitHub 使用者活動）共用程式庫。

給 asg2_01_train_gnn.ipynb（M1：RelBench 風格異質 GraphSAGE）使用，之後的實驗 notebook
（RelGNN、消融）也可以直接 import 這個檔案重用：建圖、標籤、訓練表、模型、訓練迴圈、門檻、評分。

重要規則（作業規格 §2.4、README §3）：
  * 只用 preprocess.py 產生的 8 張表。
  * row time：Users/Repositories/PullRequests/Issues/PRReviews 用 first_seen_at；
    IssueComments/PRReviewComments/Pushes 用 created_at。預測時間 t 只能看 row time < t 的列。
  * late-known 欄位（merged_at, merged_by_id, closed_at, Issues.closed_at, state_reason, dismissed_at）
    一律不放進節點特徵，merged_by_id 也不建邊（M1 最保守的作法，不會洩漏）。
  * 依時間切分：訓練 cutoff 的標籤窗必須在驗證/測試的「特徵截止時間」之前結束。
"""
from __future__ import annotations

import glob
import json
import math
import os
import random
import time as _time
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import pyarrow.compute as pc
import pyarrow.parquet as pq
import torch

ACTS = ["open_pr", "open_issue", "issue_comment", "pr_review", "pr_review_comment", "push"]
OUTPUT_COLUMNS = ["UserID"] + ACTS
TABLES = ["Users", "Repositories", "PullRequests", "Issues", "IssueComments",
          "PRReviews", "PRReviewComments", "Pushes"]
WEEK = pd.Timedelta(days=7)
# 測試：台灣 2026-11-22 00:00 = UTC 2026-11-21 16:00（表內時間皆為 UTC、無時區）
T_PRED = pd.Timestamp("2026-11-21 16:00:00")

# (pkey, row-time 欄位, {fkey: 被參照表})；merged_by_id 是 late-known FK → 不建邊
TABLE_META = {
    "Users": ("user_id", "first_seen_at", {}),
    "Repositories": ("repo_id", "first_seen_at", {}),
    "PullRequests": ("pr_id", "first_seen_at", {"repo_id": "Repositories", "author_id": "Users"}),
    "Issues": ("issue_id", "first_seen_at", {"repo_id": "Repositories", "author_id": "Users",
                                              "pull_request_id": "PullRequests", "assignee_id": "Users"}),
    "IssueComments": ("comment_id", "created_at", {"issue_id": "Issues", "repo_id": "Repositories",
                                                    "author_id": "Users"}),
    "PRReviews": ("review_id", "first_seen_at", {"pr_id": "PullRequests", "repo_id": "Repositories",
                                                  "reviewer_id": "Users"}),
    "PRReviewComments": ("comment_id", "created_at", {"pr_id": "PullRequests", "repo_id": "Repositories",
                                                       "author_id": "Users", "pull_request_review_id": "PRReviews",
                                                       "in_reply_to_id": "PRReviewComments"}),
    "Pushes": ("push_id", "created_at", {"user_id": "Users", "repo_id": "Repositories"}),
}
LATE_KNOWN = {"PullRequests": ["merged_at", "merged_by_id", "closed_at"],
              "Issues": ["closed_at", "state_reason"], "PRReviews": ["dismissed_at"]}

# 標籤來源（我的解讀，助教未公布 truth 算法）：(表, 事件時間欄, 使用者欄, 篩選)
LABEL_SOURCES = {
    "open_pr": ("PullRequests", "created_at", "author_id"),
    "open_issue": ("Issues", "created_at", "author_id"),          # 只算 is_pull_request == False
    "issue_comment": ("IssueComments", "created_at", "author_id"),
    "pr_review": ("PRReviews", "submitted_at", "reviewer_id"),
    "pr_review_comment": ("PRReviewComments", "created_at", "author_id"),
    "push": ("Pushes", "created_at", "user_id"),
}


# ----------------------------------------------------------------------------- 設定
@dataclass
class Config:
    student_id: str = "000000000"
    seed: int = 42
    smoke: bool = False
    # 資料 / 時間
    t_pred: str = "2026-11-21 16:00:00"   # UTC
    stride_days: int = 7                   # cutoff 間隔（7 = 與 t 同星期、同小時）
    max_train_cutoffs: Optional[int] = None  # None = 用全部可用週（2025-12 起）
    min_train_cutoff: str = "2025-12-01 00:00:00"  # = preprocess.py DATA_START；第一個 cutoff 為 2025-12-06 16:00 UTC
    # 取樣
    num_neighbors: Tuple[int, ...] = (64, 32)
    temporal_strategy: str = "last"        # "last" = 每種邊取時間最近的鄰居；"uniform" = 隨機
    batch_size: int = 512
    num_workers: int = 2
    # 模型
    channels: int = 128
    num_layers: int = 2
    aggr: str = "mean"
    frame_layers: int = 2                  # torch_frame ResNet 層數
    use_time_encoding: bool = True         # 相對時間編碼（Q3）
    use_global_ctx: bool = True            # cohort 全體近 7 天活動量（處理資料漂移，Q3）
    dropout: float = 0.1
    # 訓練
    epochs: int = 8
    patience: int = 3
    lr: float = 1e-3
    weight_decay: float = 0.0
    pos_weight: bool = False               # BCE 正例加權（預設關；用門檻調整處理不平衡）
    max_steps_per_epoch: Optional[int] = None
    time_budget_min: float = 55.0          # 驗證階段訓練最多幾分鐘（超過就停在目前最好的 epoch）；最終重訓的 epoch 數 ≤ 驗證階段
    final_refit: bool = True               # 驗證後把最新的週也加入訓練重訓一次
    eval_on: str = "test_users"            # 驗證時評分的使用者："test_users" 或 "cohort"


def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ----------------------------------------------------------------------------- 資料
def find_data_dir(root: str = "/kaggle/input") -> str:
    """遞迴找 Users.parquet，回傳它所在的資料夾（Dataset 叫什麼名字都可以）。
    如果找到好幾份（例如同時加了舊版與新版 Dataset），選 data_manifest.json 的 END 最晚的那份。"""
    hits = sorted(glob.glob(os.path.join(root, "**", "Users.parquet"), recursive=True))
    if not hits:
        raise FileNotFoundError(f"在 {root} 底下找不到 Users.parquet；請確認已 Add Input 你的 Dataset")
    dirs = []
    for h in hits:
        d = os.path.dirname(h)
        missing = [t for t in TABLES if not os.path.exists(os.path.join(d, f"{t}.parquet"))]
        if missing:
            print(f"[skip] {d} 缺少 {missing}"); continue
        m = find_manifest(d)
        end = json.load(open(m))["end_utc_inclusive"] if m else ""
        key = tuple(int(x) for x in end.split("-")) if end else (0,)
        dirs.append((key, d))
    if not dirs:
        raise FileNotFoundError("找到的資料夾都不完整（需要 8 個 parquet）")
    dirs.sort()
    if len(dirs) > 1:
        print("找到多份資料：", [d for _, d in dirs], "→ 使用 END 最晚的", dirs[-1][1])
    return dirs[-1][1]


def find_manifest(data_dir: str, root: Optional[str] = None) -> Optional[str]:
    cands = [os.path.join(data_dir, "data_manifest.json"), os.path.join(os.path.dirname(data_dir), "data_manifest.json")]
    cands += sorted(glob.glob(os.path.join(data_dir, "**", "data_manifest.json"), recursive=True))
    if root:
        cands += sorted(glob.glob(os.path.join(root, "**", "data_manifest.json"), recursive=True))
    for c in cands:
        if os.path.exists(c):
            return c
    return None


def data_end_exclusive(manifest_path: Optional[str], tables: Dict[str, pd.DataFrame]) -> Tuple[pd.Timestamp, str]:
    """資料結束時間（不含）。manifest 的 end_utc_inclusive = 最後一個小時檔 → +1 小時。"""
    if manifest_path:
        m = json.load(open(manifest_path))
        y, mo, d, h = m["end_utc_inclusive"].split("-")
        return pd.Timestamp(int(y), int(mo), int(d), int(h)) + pd.Timedelta(hours=1), f"manifest {manifest_path}"
    mx = max(df["_rt"].max() for df in tables.values())
    return mx.ceil("h"), "max row time (no manifest)"


def _read(path, cols):
    t = pq.read_table(path, columns=cols)
    return t


def _strlen(tab, col):
    return pc.utf8_length(tab.column(col)).to_numpy(zero_copy_only=False).astype("float64")


def _topk_cat(s: pd.Series, k: int = 30) -> pd.Series:
    s = s.astype("object").where(s.notna(), "__NA__").astype(str)
    top = s.value_counts().index[:k]
    return s.where(s.isin(top), "__OTHER__")


def _boolcat(s: pd.Series) -> pd.Series:
    return s.map({True: "T", False: "F"}).astype("object").where(s.notna(), "NA").astype(str)


def _cyc(rt: pd.Series, out: dict):
    """週期性編碼（台灣時間的小時、星期幾，sin/cos），只用這一列自己的 row time。"""
    tw = rt + pd.Timedelta(hours=8)
    h = (tw.dt.hour + tw.dt.minute / 60.0).to_numpy(dtype="float64")
    dw = tw.dt.dayofweek.to_numpy(dtype="float64")
    out["hour_sin"] = np.sin(2 * np.pi * h / 24); out["hour_cos"] = np.cos(2 * np.pi * h / 24)
    out["dow_sin"] = np.sin(2 * np.pi * dw / 7); out["dow_cos"] = np.cos(2 * np.pi * dw / 7)


def _lag_days(a: pd.Series, b: pd.Series) -> np.ndarray:
    """(a - b) 天數，log1p 壓縮；缺值 → 0，另給 is-null 指示欄。"""
    d = (a - b).dt.total_seconds().to_numpy(dtype="float64") / 86400.0
    return np.sign(np.nan_to_num(d)) * np.log1p(np.abs(np.nan_to_num(d)))


def _fk(s):
    return s.astype("float64")  # 缺值 = NaN，relbench 會略過


def load_tables(data_dir: str, cfg: Optional[Config] = None):
    """讀 8 張表，只取需要的欄位，並做「不洩漏」的特徵：
    回傳 (feat_dfs, stypes, raw)：
      feat_dfs[t]：pkey、fkeys、_rt（row time）與特徵欄
      stypes[t]：{特徵欄: torch_frame.stype}
      raw：做標籤 / baseline 用的欄位
    """
    import torch_frame
    NUM, CAT = torch_frame.numerical, torch_frame.categorical
    P = lambda t: os.path.join(data_dir, f"{t}.parquet")
    feat, stypes, raw = {}, {}, {}

    # Users
    t = _read(P("Users"), ["user_id", "github_user_id", "is_bot", "is_cohort", "first_seen_at"]).to_pandas()
    raw["Users"] = t[["user_id", "github_user_id", "is_cohort", "first_seen_at"]].copy()
    feat["Users"] = pd.DataFrame({"user_id": t.user_id.values, "_rt": t.first_seen_at,
                                  "is_bot": _boolcat(t.is_bot), "is_cohort": _boolcat(t.is_cohort)})
    stypes["Users"] = {"is_bot": CAT, "is_cohort": CAT}

    # Repositories
    tab = _read(P("Repositories"), ["repo_id", "name", "org_id", "first_seen_at"])
    t = tab.select(["repo_id", "org_id", "first_seen_at"]).to_pandas()
    feat["Repositories"] = pd.DataFrame({"repo_id": t.repo_id.values, "_rt": t.first_seen_at,
                                         "has_org": _boolcat(t.org_id.notna()),
                                         "name_len": np.log1p(np.nan_to_num(_strlen(tab, "name")))})
    stypes["Repositories"] = {"has_org": CAT, "name_len": NUM}

    # PullRequests（不含 merged_at / merged_by_id / closed_at；title、draft 只在之後有留言時才知道 → 也不用）
    tab = _read(P("PullRequests"), ["pr_id", "repo_id", "number", "author_id", "base_ref", "is_cross_repo",
                                    "created_at", "first_seen_at"])
    t = tab.to_pandas()
    raw["PullRequests"] = t[["author_id", "created_at", "first_seen_at"]].copy()
    f = {"pr_id": t.pr_id.values, "repo_id": _fk(t.repo_id), "author_id": _fk(t.author_id), "_rt": t.first_seen_at,
         "number_log": np.log1p(t.number.fillna(0).clip(lower=0).to_numpy(dtype="float64")),
         "is_cross_repo": _boolcat(t.is_cross_repo), "base_ref": _topk_cat(t.base_ref),
         "open_seen_lag": _lag_days(t.first_seen_at, t.created_at),
         "created_null": _boolcat(t.created_at.isna())}
    _cyc(t.first_seen_at, f)
    feat["PullRequests"] = pd.DataFrame(f)
    stypes["PullRequests"] = {"number_log": NUM, "is_cross_repo": CAT, "base_ref": CAT, "open_seen_lag": NUM,
                              "created_null": CAT, "hour_sin": NUM, "hour_cos": NUM, "dow_sin": NUM, "dow_cos": NUM}

    # Issues（不含 closed_at / state_reason）
    cols = ["issue_id", "repo_id", "number", "author_id", "is_pull_request", "pull_request_id", "num_labels",
            "assignee_id", "milestone_title", "issue_type", "locked", "created_via_app", "created_at", "first_seen_at"]
    tab = _read(P("Issues"), cols + ["title", "body"])
    t = tab.select(cols).to_pandas()
    raw["Issues"] = t[["author_id", "is_pull_request", "created_at", "first_seen_at"]].copy()
    f = {"issue_id": t.issue_id.values, "repo_id": _fk(t.repo_id), "author_id": _fk(t.author_id),
         "pull_request_id": _fk(t.pull_request_id), "assignee_id": _fk(t.assignee_id), "_rt": t.first_seen_at,
         "number_log": np.log1p(t.number.fillna(0).clip(lower=0).to_numpy(dtype="float64")),
         "is_pull_request": _boolcat(t.is_pull_request),
         "num_labels": np.log1p(t.num_labels.fillna(0).to_numpy(dtype="float64")),
         "has_milestone": _boolcat(t.milestone_title.notna()), "issue_type": _topk_cat(t.issue_type, 15),
         "locked": _boolcat(t.locked), "via_app": _boolcat(t.created_via_app.notna()),
         "title_len": np.log1p(np.nan_to_num(_strlen(tab, "title"))),
         "body_len": np.log1p(np.nan_to_num(_strlen(tab, "body"))),
         "open_seen_lag": _lag_days(t.first_seen_at, t.created_at)}
    _cyc(t.first_seen_at, f)
    feat["Issues"] = pd.DataFrame(f); del tab
    stypes["Issues"] = {"number_log": NUM, "is_pull_request": CAT, "num_labels": NUM, "has_milestone": CAT,
                        "issue_type": CAT, "locked": CAT, "via_app": CAT, "title_len": NUM, "body_len": NUM,
                        "open_seen_lag": NUM, "hour_sin": NUM, "hour_cos": NUM, "dow_sin": NUM, "dow_cos": NUM}

    # IssueComments
    tab = _read(P("IssueComments"), ["comment_id", "issue_id", "repo_id", "author_id", "is_pull_request",
                                     "via_app", "created_at", "body"])
    t = tab.select(["comment_id", "issue_id", "repo_id", "author_id", "is_pull_request", "via_app", "created_at"]).to_pandas()
    raw["IssueComments"] = t[["author_id", "created_at"]].copy()
    f = {"comment_id": t.comment_id.values, "issue_id": _fk(t.issue_id), "repo_id": _fk(t.repo_id),
         "author_id": _fk(t.author_id), "_rt": t.created_at, "is_pull_request": _boolcat(t.is_pull_request),
         "via_app": _boolcat(t.via_app.notna()), "body_len": np.log1p(np.nan_to_num(_strlen(tab, "body")))}
    _cyc(t.created_at, f)
    feat["IssueComments"] = pd.DataFrame(f); del tab
    stypes["IssueComments"] = {"is_pull_request": CAT, "via_app": CAT, "body_len": NUM,
                               "hour_sin": NUM, "hour_cos": NUM, "dow_sin": NUM, "dow_cos": NUM}

    # PRReviews（不含 dismissed_at）
    tab = _read(P("PRReviews"), ["review_id", "pr_id", "repo_id", "reviewer_id", "state", "submitted_at",
                                 "first_seen_at", "body"])
    t = tab.select(["review_id", "pr_id", "repo_id", "reviewer_id", "state", "submitted_at", "first_seen_at"]).to_pandas()
    raw["PRReviews"] = t[["reviewer_id", "submitted_at", "first_seen_at"]].copy()
    f = {"review_id": t.review_id.values, "pr_id": _fk(t.pr_id), "repo_id": _fk(t.repo_id),
         "reviewer_id": _fk(t.reviewer_id), "_rt": t.first_seen_at, "state": _topk_cat(t.state, 10),
         "body_len": np.log1p(np.nan_to_num(_strlen(tab, "body"))),
         "submit_seen_lag": _lag_days(t.first_seen_at, t.submitted_at)}
    _cyc(t.first_seen_at, f)
    feat["PRReviews"] = pd.DataFrame(f); del tab
    stypes["PRReviews"] = {"state": CAT, "body_len": NUM, "submit_seen_lag": NUM,
                           "hour_sin": NUM, "hour_cos": NUM, "dow_sin": NUM, "dow_cos": NUM}

    # PRReviewComments
    cols = ["comment_id", "pr_id", "repo_id", "author_id", "pull_request_review_id", "in_reply_to_id",
            "file_extension", "subject_type", "position", "original_position", "created_at"]
    tab = _read(P("PRReviewComments"), cols + ["body", "diff_hunk"])
    t = tab.select(cols).to_pandas()
    raw["PRReviewComments"] = t[["author_id", "created_at"]].copy()
    f = {"comment_id": t.comment_id.values, "pr_id": _fk(t.pr_id), "repo_id": _fk(t.repo_id),
         "author_id": _fk(t.author_id), "pull_request_review_id": _fk(t.pull_request_review_id),
         "in_reply_to_id": _fk(t.in_reply_to_id), "_rt": t.created_at,
         "file_extension": _topk_cat(t.file_extension, 30), "subject_type": _topk_cat(t.subject_type, 5),
         "position_null": _boolcat(t.position.isna()),
         "orig_pos_log": np.log1p(t.original_position.fillna(0).clip(lower=0).to_numpy(dtype="float64")),
         "body_len": np.log1p(np.nan_to_num(_strlen(tab, "body"))),
         "hunk_len": np.log1p(np.nan_to_num(_strlen(tab, "diff_hunk")))}
    _cyc(t.created_at, f)
    feat["PRReviewComments"] = pd.DataFrame(f); del tab
    stypes["PRReviewComments"] = {"file_extension": CAT, "subject_type": CAT, "position_null": CAT,
                                  "orig_pos_log": NUM, "body_len": NUM, "hunk_len": NUM,
                                  "hour_sin": NUM, "hour_cos": NUM, "dow_sin": NUM, "dow_cos": NUM}

    # Pushes
    t = _read(P("Pushes"), ["push_id", "user_id", "repo_id", "ref_type", "branch", "is_new_ref", "created_at"]).to_pandas()
    raw["Pushes"] = t[["user_id", "created_at"]].copy()
    f = {"push_id": t.push_id.values, "user_id": _fk(t.user_id), "repo_id": _fk(t.repo_id), "_rt": t.created_at,
         "ref_type": _topk_cat(t.ref_type, 5), "branch": _topk_cat(t.branch, 30), "is_new_ref": _boolcat(t.is_new_ref)}
    _cyc(t.created_at, f)
    feat["Pushes"] = pd.DataFrame(f); del t
    stypes["Pushes"] = {"ref_type": CAT, "branch": CAT, "is_new_ref": CAT,
                        "hour_sin": NUM, "hour_cos": NUM, "dow_sin": NUM, "dow_cos": NUM}

    for k, df in feat.items():
        pk = TABLE_META[k][0]
        assert (df[pk].values == np.arange(len(df))).all(), f"{k}: PK 不是 0..n-1"
        assert df["_rt"].notna().all(), f"{k}: row time 有缺值"
        for c in LATE_KNOWN.get(k, []):
            assert c not in df.columns
    return feat, stypes, raw


def build_graph(feat, stypes, cache_dir=None):
    """RelBench Database → make_pkey_fkey_graph：每個 FK 一種邊 f2p_<fk> 與反向 rev_f2p_<fk>。"""
    from relbench.base import Database, Table
    from relbench.modeling.graph import make_pkey_fkey_graph
    tables = {}
    for k, df in feat.items():
        pk, _, fks = TABLE_META[k]
        tables[k] = Table(df=df, fkey_col_to_pkey_table=dict(fks), pkey_col=pk, time_col="_rt")
    db = Database(tables)
    data, col_stats = make_pkey_fkey_graph(db, {k: dict(v) for k, v in stypes.items()}, cache_dir=cache_dir)
    return data, col_stats


# ----------------------------------------------------------------------------- 標籤 / baseline / 情境特徵
class LabelIndex:
    """每個 activity：依事件時間排序的 (user_id, event_time, row_time) 陣列，方便快速查任何時間窗。"""

    def __init__(self, raw):
        self.src = {}
        iss = raw["Issues"]
        for a, (tname, tcol, ucol) in LABEL_SOURCES.items():
            df = raw[tname]
            if a == "open_issue":
                df = iss[~iss.is_pull_request.fillna(False).astype(bool)]
            df = df[df[ucol].notna() & df[tcol].notna()]
            rt_col = TABLE_META[tname][1]
            et = df[tcol].values.astype("datetime64[ns]")
            rt = df[rt_col].values.astype("datetime64[ns]")
            o = np.argsort(et, kind="stable")
            self.src[a] = (df[ucol].values.astype("int64")[o], et[o], rt[o])

    def labels(self, users: np.ndarray, start, end, known_before=None) -> np.ndarray:
        """users 在 [start, end)（事件時間）是否做了每個 activity；known_before 不為 None 時，
        只算 row time < known_before 的列（= 那個時間點真的看得到的資料）。"""
        start, end = np.datetime64(pd.Timestamp(start)), np.datetime64(pd.Timestamp(end))
        out = np.zeros((len(users), len(ACTS)), dtype=np.float32)
        for j, a in enumerate(ACTS):
            u, et, rt = self.src[a]
            lo, hi = np.searchsorted(et, start), np.searchsorted(et, end)
            uu = u[lo:hi]
            if known_before is not None:
                uu = uu[rt[lo:hi] < np.datetime64(pd.Timestamp(known_before))]
            out[:, j] = np.isin(users, uu)
        return out

    def global_ctx(self, cohort: np.ndarray, seed_time) -> np.ndarray:
        """cohort 全體在 seed_time 前 7 天的活動量（每個 activity：事件數、活躍人數，log1p）。
        只用 row time < seed_time 的列 → 不洩漏；用來告訴模型「這週資料量的量級」（GH Archive 缺漏 / 漂移）。"""
        st = np.datetime64(pd.Timestamp(seed_time)); s0 = st - np.timedelta64(7, "D")
        v = []
        for a in ACTS:
            u, et, rt = self.src[a]
            lo, hi = np.searchsorted(et, s0), np.searchsorted(et, st)
            m = rt[lo:hi] < st
            uu = u[lo:hi][m]
            uu = uu[np.isin(uu, cohort)]
            v += [math.log1p(len(uu)), math.log1p(len(np.unique(uu)))]
        return np.asarray(v, dtype=np.float32)


def weekly_cutoffs(first: pd.Timestamp, last: pd.Timestamp, anchor: pd.Timestamp, stride_days: int = 7):
    """與 anchor（= t）同相位的 cutoff 序列（stride_days 天一個），介於 [first, last]。"""
    step = pd.Timedelta(days=stride_days)
    k_hi = math.floor((last - anchor) / step)
    k_lo = math.ceil((first - anchor) / step)
    return [anchor + k * step for k in range(k_lo, k_hi + 1)]


@dataclass
class TimePlan:
    t_pred: pd.Timestamp
    end_excl: pd.Timestamp
    gap: pd.Timedelta
    val_cutoff: pd.Timestamp
    calib_cutoff: pd.Timestamp
    train_cutoffs: List[pd.Timestamp]          # 驗證階段的訓練 cutoff
    final_train_cutoffs: List[pd.Timestamp]    # 最終重訓的訓練 cutoff

    def seed(self, cutoff):
        """特徵截止時間 = cutoff - gap - 1 秒（取樣含等號，所以減 1 秒讓「看得到的列」嚴格 < cutoff - gap）。"""
        return cutoff - self.gap - pd.Timedelta(seconds=1)

    def describe(self):
        f = lambda x: f"{x:%Y-%m-%d %H:%M} UTC (TW {x + pd.Timedelta(hours=8):%m-%d %H:%M})"
        lines = [f"prediction t      : {f(self.t_pred)}",
                 f"data end (excl.)  : {f(self.end_excl)}",
                 f"gap t - data end  : {self.gap}",
                 f"test feature time : {f(self.seed(self.t_pred))}",
                 f"val cutoff        : {f(self.val_cutoff)}  label window -> {f(self.val_cutoff + WEEK)}",
                 f"calib cutoff      : {f(self.calib_cutoff)}  (threshold tuning week)",
                 f"train cutoffs     : {len(self.train_cutoffs)}  {self.train_cutoffs[0]:%Y-%m-%d} .. {self.train_cutoffs[-1]:%Y-%m-%d}",
                 f"final cutoffs     : {len(self.final_train_cutoffs)}  {self.final_train_cutoffs[0]:%Y-%m-%d} .. {self.final_train_cutoffs[-1]:%Y-%m-%d}"]
        return "\n".join(lines)


def make_time_plan(cfg: Config, end_excl: pd.Timestamp, val_offset: int = 0) -> TimePlan:
    """val_offset = 0：val = 最後一個完整週；k > 0：往前 k 週（多週回測用，訓練 cutoff 也跟著往前）。"""
    t = pd.Timestamp(cfg.t_pred)
    gap = t - end_excl
    assert gap >= pd.Timedelta(0), "資料結尾比預測時間還晚？請檢查 END"
    first = pd.Timestamp(cfg.min_train_cutoff)
    # 標籤窗必須完整落在資料內：cutoff + 7d <= end_excl
    allc = weekly_cutoffs(first, end_excl - WEEK, t, cfg.stride_days)
    weekly = [c for c in allc if ((t - c) / pd.Timedelta(days=1)) % 7 == 0]
    val_c = weekly[-1 - val_offset]; calib_c = val_c - WEEK
    # 時間切分：訓練樣本的標籤窗必須在「被評估那週的特徵截止時間」之前結束
    def ok(c, eval_c):
        return c + WEEK <= eval_c - gap
    train = [c for c in allc if ok(c, calib_c)]
    final = [c for c in allc if ok(c, t)]
    if cfg.max_train_cutoffs:
        train, final = train[-cfg.max_train_cutoffs:], final[-cfg.max_train_cutoffs:]
    assert train, "沒有可用的訓練 cutoff"
    return TimePlan(t, end_excl, gap, val_c, calib_c, train, final)


@dataclass
class SampleTable:
    nodes: torch.Tensor   # Users 的 row index
    time: torch.Tensor    # seed time（unix 秒）
    y: torch.Tensor       # [N, 6]（測試時全 0）
    ctx: torch.Tensor     # [N, C]
    cutoff: np.ndarray    # 每列的 cutoff（字串）


def to_unix(ts) -> int:
    return int(pd.Timestamp(ts).value // 10**9)


def make_sample_table(users: np.ndarray, cutoffs, plan: TimePlan, li: LabelIndex, cohort: np.ndarray,
                      with_labels=True, seed_gap: Optional[pd.Timedelta] = None) -> SampleTable:
    """seed_gap=None → 用 plan.gap（正常情況）；給 pd.Timedelta(0) 等 = 消融實驗「不模擬 gap」。"""
    N, Y, T, C, K = [], [], [], [], []
    for c in cutoffs:
        st = plan.seed(c) if seed_gap is None else c - seed_gap - pd.Timedelta(seconds=1)
        N.append(users); T.append(np.full(len(users), to_unix(st), dtype=np.int64))
        Y.append(li.labels(users, c, c + WEEK) if with_labels else np.zeros((len(users), len(ACTS)), np.float32))
        C.append(np.repeat(li.global_ctx(cohort, st)[None, :], len(users), 0))
        K.append(np.full(len(users), str(c)))
    return SampleTable(torch.from_numpy(np.concatenate(N)), torch.from_numpy(np.concatenate(T)),
                       torch.from_numpy(np.concatenate(Y)), torch.from_numpy(np.concatenate(C)), np.concatenate(K))


class AttachYCtx:
    def __init__(self, entity, y, ctx):
        self.entity, self.y, self.ctx = entity, y, ctx

    def __call__(self, batch):
        idx = batch[self.entity].input_id
        batch[self.entity].y = self.y[idx]
        batch[self.entity].ctx = self.ctx[idx]
        return batch


def make_loader(data, st: SampleTable, cfg: Config, shuffle: bool, entity="Users", transform=None):
    from torch_geometric.loader import NeighborLoader
    g = torch.Generator(); g.manual_seed(cfg.seed)
    return NeighborLoader(
        data, num_neighbors=list(cfg.num_neighbors), time_attr="time",
        input_nodes=(entity, st.nodes), input_time=st.time, temporal_strategy=cfg.temporal_strategy,
        transform=transform or AttachYCtx(entity, st.y, st.ctx), batch_size=cfg.batch_size, shuffle=shuffle,
        num_workers=cfg.num_workers, persistent_workers=(cfg.num_workers > 0 and shuffle), generator=g)


# ----------------------------------------------------------------------------- 模型
class RDLModel(torch.nn.Module):
    """RelBench 風格：torch_frame 欄位編碼 + 相對時間編碼 → HeteroGraphSAGE → 6 個 sigmoid 輸出。"""

    def __init__(self, data, col_stats, cfg: Config, ctx_dim: int, entity="Users", gnn_cls=None, dyn_dims=None):
        super().__init__()
        from relbench.modeling.nn import HeteroEncoder, HeteroTemporalEncoder, HeteroGraphSAGE
        from torch_geometric.nn import MLP
        self.entity, self.cfg = entity, cfg
        self.encoder = HeteroEncoder(
            channels=cfg.channels,
            node_to_col_names_dict={nt: data[nt].tf.col_names_dict for nt in data.node_types},
            node_to_col_stats=col_stats,
            torch_frame_model_kwargs={"channels": cfg.channels, "num_layers": cfg.frame_layers})
        self.temporal = HeteroTemporalEncoder([nt for nt in data.node_types if "time" in data[nt]], cfg.channels)
        gnn_cls = gnn_cls or HeteroGraphSAGE
        self.gnn = gnn_cls(node_types=data.node_types, edge_types=data.edge_types, channels=cfg.channels,
                           aggr=cfg.aggr, num_layers=cfg.num_layers)
        self.ctx_dim = ctx_dim if cfg.use_global_ctx else 0
        if self.ctx_dim:
            self.ctx_mlp = MLP([ctx_dim, 32, 32], norm=None, dropout=0.0)
        self.head = MLP([cfg.channels + (32 if self.ctx_dim else 0), cfg.channels, len(ACTS)],
                        norm=None, dropout=cfg.dropout)
        # 選用：每個 batch 動態算出的節點特徵（例如 late-known 欄位「在 seed time 前是否已發生」），見 asg2_exp.py
        self.dyn = torch.nn.ModuleDict({nt: torch.nn.Linear(d, cfg.channels) for nt, d in (dyn_dims or {}).items()}) \
            if dyn_dims else None
        self.register_buffer("ctx_mean", torch.zeros(max(ctx_dim, 1)))
        self.register_buffer("ctx_std", torch.ones(max(ctx_dim, 1)))

    def forward(self, batch):
        e = self.entity
        seed_time = batch[e].seed_time
        x = self.encoder(batch.tf_dict)
        if self.cfg.use_time_encoding:
            rel = self.temporal(seed_time, batch.time_dict, batch.batch_dict)
            for nt, r in rel.items():
                x[nt] = x[nt] + r
        if self.dyn is not None:
            for nt, lin in self.dyn.items():
                if nt in x and "dyn" in batch[nt]:
                    x[nt] = x[nt] + lin(batch[nt].dyn)
        x = self.gnn(x, batch.edge_index_dict)
        h = x[e][: seed_time.size(0)]
        if self.ctx_dim:
            c = (batch[e].ctx - self.ctx_mean) / self.ctx_std
            h = torch.cat([h, self.ctx_mlp(c)], -1)
        return self.head(h)


# ----------------------------------------------------------------------------- 評分（與官方 evaluate.py 相同的 F1 定義）
def f1(tp, fp, fn):
    return 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 1.0


def score_arrays(pred: np.ndarray, truth: np.ndarray) -> dict:
    pred, truth = pred.astype(int), truth.astype(int)
    tp = ((pred == 1) & (truth == 1)).sum(0); fp = ((pred == 1) & (truth == 0)).sum(0)
    fn = ((pred == 0) & (truth == 1)).sum(0)
    per = {a: float(f1(*c)) for a, c in zip(ACTS, zip(tp, fp, fn))}
    macro = sum(per.values()) / len(ACTS); micro = float(f1(tp.sum(), fp.sum(), fn.sum()))
    return {"per_event": per, "macro_f1": macro, "micro_f1": micro, "score": (macro + micro) / 2}


def tune_thresholds(prob: np.ndarray, truth: np.ndarray, grid=None) -> np.ndarray:
    """每個 activity 各自找讓 F1 最大的門檻（規格分數 = (macro-F1 + micro-F1)/2）。"""
    grid = np.round(np.arange(0.02, 0.96, 0.01), 2) if grid is None else grid
    th = np.zeros(prob.shape[1])
    for j in range(prob.shape[1]):
        best, bt = -1, 0.5
        for g in grid:
            p = prob[:, j] >= g
            tp = int((p & (truth[:, j] == 1)).sum()); fp = int((p & (truth[:, j] == 0)).sum())
            fn = int((~p & (truth[:, j] == 1)).sum())
            v = f1(tp, fp, fn)
            if v > best + 1e-12:
                best, bt = v, g
        th[j] = bt
    return th


def tune_rates(prob: np.ndarray, truth: np.ndarray) -> np.ndarray:
    """每個 activity 找讓 F1 最大的「預測為 1 的比例」（= 依機率排序取前 k 名）。
    與機率門檻相比，比例對「模型機率刻度在不同週漂移」比較穩健（實測見 run_summary）。"""
    n = len(prob); rates = np.zeros(prob.shape[1])
    for j in range(prob.shape[1]):
        o = np.argsort(-prob[:, j], kind="stable"); t = truth[o, j].astype(int)
        P = int(t.sum())
        if P == 0:
            rates[j] = 0.0; continue                       # 沒有正例：全部猜 0 時 F1 = 1（官方定義）
        tp = np.cumsum(t); k = np.arange(1, n + 1)
        f = 2 * tp / (k + P)                               # 2TP / (2TP + FP + FN) = 2TP / (k + P)
        rates[j] = (int(np.argmax(f)) + 1) / n
    return rates


def apply_rates(prob: np.ndarray, rates: np.ndarray) -> np.ndarray:
    """依比例取前 k 名為 1（k = round(rate × 人數)），回傳 0/1 矩陣。"""
    out = np.zeros(prob.shape, dtype=int)
    for j in range(prob.shape[1]):
        k = int(round(rates[j] * len(prob)))
        if k > 0:
            out[np.argsort(-prob[:, j], kind="stable")[:k], j] = 1
    return out


def fmt_scores(name, s):
    per = " ".join(f"{a}={100*s['per_event'][a]:5.1f}" for a in ACTS)
    return f"{name:<28} Score={100*s['score']:6.2f}  macro={100*s['macro_f1']:6.2f}  micro={100*s['micro_f1']:6.2f} | {per}"


# ----------------------------------------------------------------------------- 訓練 / 推論
@torch.no_grad()
def predict(model, loader, device) -> Tuple[np.ndarray, np.ndarray]:
    model.eval(); P, Y = [], []
    for b in loader:
        b = b.to(device)
        P.append(torch.sigmoid(model(b)).float().cpu()); Y.append(b["Users"].y.cpu())
    return torch.cat(P).numpy(), torch.cat(Y).numpy()


def train_epoch(model, loader, opt, device, pos_weight=None, max_steps=None, log_every=200):
    model.train(); tot, n, t0 = 0.0, 0, _time.time()
    lossf = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    for i, b in enumerate(loader):
        if max_steps and i >= max_steps:
            break
        b = b.to(device)
        opt.zero_grad()
        loss = lossf(model(b), b["Users"].y)
        loss.backward(); opt.step()
        tot += loss.item() * b["Users"].y.size(0); n += b["Users"].y.size(0)
        if log_every and (i + 1) % log_every == 0:
            print(f"    step {i+1}: loss {tot/n:.4f}  ({_time.time()-t0:.0f}s)", flush=True)
    return tot / max(n, 1)


def match_rate_thresholds(p_new: np.ndarray, rates: np.ndarray) -> np.ndarray:
    """讓新模型在同一批使用者上的「預測為 1 的比例」等於 rates（每個 activity 一個）。
    用途：最終重訓的模型機率刻度和驗證階段的模型不同，直接沿用門檻會偏掉；
    所以沿用「驗證過的模型 + 驗證門檻」在測試使用者上的預測比例，再換算成新模型的門檻。"""
    th = np.zeros(p_new.shape[1])
    for j in range(p_new.shape[1]):
        k = int(round(rates[j] * len(p_new)))
        if k <= 0:
            th[j] = 1.0 + 1e-6          # 全部預測 0
        else:
            th[j] = float(np.sort(p_new[:, j])[::-1][k - 1])
    return th


def copy_last_week(li: LabelIndex, users, cutoff, feature_time=None):
    """README 的 Copy last week：cutoff 前 7 天做過 → 1。
    feature_time 給定時，只用 row time < feature_time 的列，且窗改成 [feature_time-7d, feature_time)（模擬資料空窗）。"""
    if feature_time is None:
        return li.labels(users, cutoff - WEEK, cutoff)
    return li.labels(users, feature_time - WEEK, feature_time, known_before=feature_time)


def versions() -> dict:
    import sys, platform
    v = {"python": sys.version.split()[0], "platform": platform.platform(), "numpy": np.__version__,
         "pandas": pd.__version__, "torch": torch.__version__, "cuda": torch.version.cuda}
    for m in ["torch_geometric", "pyg_lib", "torch_frame", "relbench", "pyarrow"]:
        try:
            v[m] = __import__(m).__version__
        except Exception as e:  # noqa
            v[m] = f"not available ({type(e).__name__})"
    return v


In [ ]:
%%writefile asg2_exp.py
# -*- coding: utf-8 -*-
"""asg2_exp.py — Assignment 2 實驗（Q2 / Q3 / Q4 消融與對照）共用程式。

給 asg2_02_experiments.ipynb 使用；建立在 asg2_lib.py（M1 的資料、圖、模型）之上。
每個實驗 = 一個有名字的設定（ExpSpec）。所有設定用同一套評估流程：
  * 訓練 cutoff 的標籤窗都在 calib 週的特徵截止時間之前結束（時間切分、模擬 gap）。
  * epoch 選擇：用 calib 週的平均 AP（不看 val 週）。
  * 決策規則：在 calib 週找每個活動讓 F1 最大的「預測為 1 的比例」，套到 val 週（= honest Score）。
  * 同時記錄：calib 機率門檻、固定 0.5、val 週自己調比例（偏樂觀）——用來做「決策規則」消融。
結果一列一列 append 到 results.csv；重跑時已完成的設定會跳過。
"""
from __future__ import annotations

import copy
import json
import math
import os
import time
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import torch

import asg2_lib as L

FACT_TABLES = ["PullRequests", "Issues", "IssueComments", "PRReviews", "PRReviewComments", "Pushes"]
CYCLIC_COLS = ["hour_sin", "hour_cos", "dow_sin", "dow_cos"]
LATE_COLS = {"PullRequests": ["merged_at", "closed_at"], "Issues": ["closed_at"], "PRReviews": ["dismissed_at"]}
LECTURE, DESIGN, BASELINE = "lecture", "design (ours)", "baseline"


# ============================================================================ 實驗設定
@dataclass
class ExpSpec:
    name: str
    group: str                      # "Q2" / "Q3" / "Q4" / "baseline"
    basis: str                      # LECTURE / DESIGN / BASELINE
    ref: str                        # 講義頁碼或論文
    desc: str                       # 英文短描述（會出現在圖上）
    cfg: Dict = field(default_factory=dict)        # 覆蓋 asg2_lib.Config
    gnn: str = "sage"               # "sage" / "relgnn" / "gat"
    drop_tables: tuple = ()
    keep_tables: tuple = ()
    drop_rev: bool = False
    drop_fks: tuple = ()            # 例如 ("in_reply_to_id",)
    drop_cols: tuple = ()           # 例如 CYCLIC_COLS
    lookback_days: Optional[float] = None
    late_dynamic: bool = False
    no_gap_sim: bool = False
    val_offset: int = 0
    seed: int = 42
    kind: str = "gnn"               # "gnn" / "lgbm"
    compare_to: str = "M1"          # 消融圖上跟誰比

    def graph_key(self):
        return json.dumps([sorted(self.drop_tables), sorted(self.keep_tables), self.drop_rev,
                           sorted(self.drop_fks), sorted(self.drop_cols)])


def default_specs() -> List[ExpSpec]:
    """完整實驗清單（依優先順序）。名稱就是 RUN_LIST 裡用的字串。"""
    S = []
    # ---- Q4 主結果：M1、M2 各 3 個 seed；tabular baseline
    for sd in (42, 43, 44):
        S.append(ExpSpec(f"M1_s{sd}", "Q4", LECTURE, "L5-1 p22-p29, p35; L5-2 p20 (RelBench GNN)",
                         "M1 HeteroGraphSAGE (RelBench)", seed=sd))
    for sd in (42, 43, 44):
        S.append(ExpSpec(f"M2_RelGNN_s{sd}", "Q2", LECTURE, "L5-2 p6-p9; arXiv:2502.06784 (official code)",
                         "M2 RelGNN atomic routes", gnn="relgnn", cfg={"aggr": "sum"}, seed=sd))
    S.append(ExpSpec("LightGBM_counts", "Q4", BASELINE, "L5-1 p10, p30 (hand-engineered features); L5-2 p8",
                     "LightGBM on COUNT/recency features", kind="lgbm"))
    # ---- Q3 時間
    S += [
        ExpSpec("Q3_no_time_enc", "Q3", LECTURE, "L5-1 p31, p35; L5-2 p11-p12; README 35.70→54.82",
                "no relative time encoding", cfg={"use_time_encoding": False}),
        ExpSpec("Q3_uniform_sampling", "Q3", LECTURE, "L5-1 p29, p35 (temporal sampling)",
                "temporal_strategy=uniform", cfg={"temporal_strategy": "uniform"}),
        ExpSpec("Q3_lookback_7d", "Q3", LECTURE, "L5-1 p29 (time-dependent computation graph)",
                "neighbors only from last 7 days", lookback_days=7),
        ExpSpec("Q3_lookback_28d", "Q3", LECTURE, "L5-1 p29", "neighbors only from last 28 days", lookback_days=28),
        ExpSpec("Q3_lookback_91d", "Q3", LECTURE, "L5-1 p29", "neighbors only from last 91 days", lookback_days=91),
        ExpSpec("Q3_no_gap_sim", "Q3", DESIGN, "ours (data end < t; gap = t - end of data)",
                "no gap simulation in training", no_gap_sim=True),
        ExpSpec("Q3_no_cyclic", "Q3", DESIGN, "ours (cf. L5-1 p35 seasonal features)",
                "no hour/day-of-week features", drop_cols=tuple(CYCLIC_COLS)),
        ExpSpec("Q3_no_global_ctx", "Q3", DESIGN, "ours (GH Archive volume drift)",
                "no cohort-activity context", cfg={"use_global_ctx": False}),
        ExpSpec("Q3_late_dynamic", "Q3", LECTURE, "spec §2.4; L5-1 p10 (point-in-time correctness)",
                "late-known cols used only if < seed time", late_dynamic=True),
    ]
    # ---- Q2 關係
    for t in FACT_TABLES + ["Repositories"]:
        S.append(ExpSpec(f"Q2_drop_{t}", "Q2", LECTURE, "L5-1 p15-p19, p32-p34",
                         f"leave out table {t}", drop_tables=(t,)))
    S += [
        ExpSpec("Q2_users_pushes_only", "Q2", LECTURE, "L5-1 p15-p19", "only Users + Pushes",
                keep_tables=("Users", "Pushes")),
        ExpSpec("Q2_no_reverse_edges", "Q2", LECTURE, "L5-1 p19 (PK-FK links), p28", "no reverse (pkey→fkey) edges",
                drop_rev=True),
        ExpSpec("Q2_no_self_loop_fk", "Q2", LECTURE, "L5-1 p18 (schema graph)", "no in_reply_to_id self-FK",
                drop_fks=("in_reply_to_id",)),
        ExpSpec("Q2_layers_1", "Q2", LECTURE, "L5-1 p32-p34 (multi-hop)", "1 GNN layer",
                cfg={"num_layers": 1, "num_neighbors": [64]}),
        ExpSpec("Q2_layers_3", "Q2", LECTURE, "L5-1 p32-p34 (multi-hop)", "3 GNN layers",
                cfg={"num_layers": 3, "num_neighbors": [64, 32, 16]}),
        ExpSpec("Q2_sum_aggr", "Q2", LECTURE, "L5-1 p30-p31 (learnable JOIN+AGG); L5-2 p4", "sum aggregation",
                cfg={"aggr": "sum"}),
        ExpSpec("Q2_HeteroGAT", "Q2", LECTURE, "L5-2 p4; RelBench HeteroGAT", "HeteroGAT (attention)", gnn="gat"),
    ]
    # ---- Q4 多週回測（每個往前推 1–3 週都重新訓練）
    for k in (1, 2, 3):
        S.append(ExpSpec(f"M1_s42_val-{k}w", "Q4", LECTURE, "L5-1 p21 (time split)", f"M1, val week -{k}", val_offset=k))
        S.append(ExpSpec(f"M2_RelGNN_s42_val-{k}w", "Q4", LECTURE, "L5-2 p6-p9", f"M2, val week -{k}",
                         gnn="relgnn", cfg={"aggr": "sum"}, val_offset=k))
        S.append(ExpSpec(f"LightGBM_counts_val-{k}w", "Q4", BASELINE, "L5-1 p30", f"LightGBM, val week -{k}",
                         kind="lgbm", val_offset=k))
    return S


# ============================================================================ 圖的變體
def derive_graph(base, base_stats, spec: ExpSpec, late: Optional[Dict] = None):
    """從完整的圖拿掉表 / 邊 / 欄位，不用重新建圖（省時間）。"""
    from torch_geometric.data import HeteroData
    from relbench.modeling.graph import drop_tensor_frame_columns
    keep = set(spec.keep_tables) if spec.keep_tables else set(base.node_types) - set(spec.drop_tables)
    assert "Users" in keep
    d, stats = HeteroData(), {}
    for nt in base.node_types:
        if nt not in keep:
            continue
        tf, st = base[nt].tf, base_stats[nt]
        names = [c for v in tf.col_names_dict.values() for c in v]
        dc = [c for c in spec.drop_cols if c in names]
        if dc:
            tf, st = drop_tensor_frame_columns(tf, st, dc)
        d[nt].tf, d[nt].time, stats[nt] = tf, base[nt].time, st
        if late and nt in late:
            d[nt].late = late[nt]
    bad = {f"f2p_{fk}" for fk in spec.drop_fks} | {f"rev_f2p_{fk}" for fk in spec.drop_fks}
    for et in base.edge_types:
        s, r, t = et
        if s not in keep or t not in keep or r in bad or (spec.drop_rev and r.startswith("rev_")):
            continue
        d[et].edge_index = base[et].edge_index
    return d, stats


def load_late(data_dir: str) -> Dict[str, torch.Tensor]:
    """late-known 時間欄（UNIX 秒，float64；缺值 = NaN）。只給 Q3_late_dynamic 用，而且只在 batch 裡跟 seed time 比較，
    所以「只有早於 seed time 的值才會被模型看到」（規格 §2.4）。"""
    out = {}
    for t, cols in LATE_COLS.items():
        df = pd.read_parquet(os.path.join(data_dir, f"{t}.parquet"), columns=cols)
        arr = np.stack([np.where(df[c].notna(), df[c].values.astype("datetime64[s]").astype("int64").astype("float64"),
                                 np.nan) for c in cols], 1)
        out[t] = torch.from_numpy(arr)
    return out


class ExpTransform(L.AttachYCtx):
    """在 L.AttachYCtx 之外：(1) lookback：拿掉比 seed time 早 lookback_days 以上的事實表鄰居邊；
    (2) late_dynamic：late-known 欄位只在「< seed time」時才當特徵（是否已發生、距今幾天）。"""

    def __init__(self, entity, y, ctx, lookback_days=None, late_dynamic=False):
        super().__init__(entity, y, ctx)
        self.lookback, self.late_dynamic = lookback_days, late_dynamic

    def __call__(self, batch):
        batch = super().__call__(batch)
        seed = batch[self.entity].seed_time
        if self.lookback:
            lim = float(self.lookback) * 86400.0
            for et in batch.edge_types:
                s = et[0]
                if s not in FACT_TABLES:
                    continue
                ei = batch[et].edge_index
                if ei.numel() == 0:
                    continue
                age = (seed[batch[s].batch[ei[0]]] - batch[s].time[ei[0]]).double()
                batch[et].edge_index = ei[:, age <= lim]
        if self.late_dynamic:
            for nt in LATE_COLS:
                if nt in batch.node_types and "late" in batch[nt]:
                    lt = batch[nt].late
                    st = seed[batch[nt].batch].double().unsqueeze(1)
                    known = (~torch.isnan(lt)) & (lt < st)
                    age = torch.where(known, torch.log1p(((st - lt) / 86400.0).clamp(min=0)), torch.zeros_like(lt))
                    batch[nt].dyn = torch.cat([known.float(), age.float()], 1)
                    del batch[nt].late
        return batch


# ============================================================================ GNN 選擇
class HeteroGraphSAGEKeep(torch.nn.Module):
    """與 RelBench HeteroGraphSAGE 相同，但「沒有任何入邊的節點類型」保留上一層的表示。
    只用在 Q2_no_reverse_edges：拿掉反向邊後，事實表（Pushes 等）收不到訊息，原版會在第 2 層因 x=None 出錯。"""

    def __init__(self, node_types, edge_types, channels, aggr="mean", num_layers=2):
        super().__init__()
        from relbench.modeling.nn import HeteroGraphSAGE
        self.inner = HeteroGraphSAGE(node_types=node_types, edge_types=edge_types, channels=channels, aggr=aggr,
                                     num_layers=num_layers)

    def forward(self, x_dict, edge_index_dict, *args, **kw):
        for conv, norm_dict in zip(self.inner.convs, self.inner.norms):
            out = conv(x_dict, edge_index_dict)
            out = {k: norm_dict[k](v).relu() for k, v in out.items()}
            x_dict = {**x_dict, **out}
        return x_dict


def gnn_factory(kind: str, heads: int = 4):
    if kind == "sage":
        return None                                   # asg2_lib 預設 HeteroGraphSAGE
    if kind == "sage_keep":
        return lambda node_types, edge_types, channels, aggr, num_layers: HeteroGraphSAGEKeep(
            node_types, edge_types, channels, aggr, num_layers)
    if kind == "gat":
        from relbench.modeling.nn import HeteroGAT
        return lambda node_types, edge_types, channels, aggr, num_layers: HeteroGAT(
            node_types=node_types, edge_types=edge_types, channels=channels, heads=heads, num_layers=num_layers)
    if kind == "relgnn":
        from relgnn_vendor import RelGNN, get_atomic_routes
        return lambda node_types, edge_types, channels, aggr, num_layers: RelGNN(
            node_types=node_types, edge_types=get_atomic_routes(edge_types), channels=channels, aggr=aggr,
            num_model_layers=num_layers, num_heads=heads, simplified_MP=True)
    raise ValueError(kind)


# ============================================================================ 評估
def eval_block(p_cal, y_cal, p_val, y_val) -> dict:
    from sklearn.metrics import average_precision_score, roc_auc_score
    r_cal = L.tune_rates(p_cal, y_cal)
    s_rate = L.score_arrays(L.apply_rates(p_val, r_cal), y_val)
    s_pth = L.score_arrays(p_val >= L.tune_thresholds(p_cal, y_cal), y_val)
    s05 = L.score_arrays(p_val >= 0.5, y_val)
    s_orc = L.score_arrays(L.apply_rates(p_val, L.tune_rates(p_val, y_val)), y_val)
    row = {"score": s_rate["score"], "macro_f1": s_rate["macro_f1"], "micro_f1": s_rate["micro_f1"],
           **{f"f1_{a}": s_rate["per_event"][a] for a in L.ACTS},
           "score_calib_probthr": s_pth["score"], "score_th05": s05["score"], "score_val_rate_optimistic": s_orc["score"]}
    aucs, aps = [], []
    for j in range(len(L.ACTS)):
        if 0 < y_val[:, j].sum() < len(y_val):
            aucs.append(roc_auc_score(y_val[:, j], p_val[:, j])); aps.append(average_precision_score(y_val[:, j], p_val[:, j]))
    row["mean_auc"] = float(np.mean(aucs)) if aucs else np.nan
    row["mean_ap"] = float(np.mean(aps)) if aps else np.nan
    return row


def mean_ap(p, y):
    from sklearn.metrics import average_precision_score
    v = [average_precision_score(y[:, j], p[:, j]) for j in range(y.shape[1]) if y[:, j].sum() > 0]
    return float(np.mean(v)) if v else 0.0


def baseline_rows(li, eval_u, plan) -> List[dict]:
    y = li.labels(eval_u, plan.val_cutoff, plan.val_cutoff + L.WEEK)
    out = []
    for name, pred, desc in [
        ("All zeros", np.zeros_like(y), "predict all 0"),
        ("Copy last week", L.copy_last_week(li, eval_u, plan.val_cutoff), "README: did it in the 7 days before t"),
        ("Copy last week (gap)", L.copy_last_week(li, eval_u, plan.val_cutoff, plan.seed(plan.val_cutoff)),
         "same, but only data visible at the feature cutoff"),
    ]:
        s = L.score_arrays(pred, y)
        out.append({"name": name, "group": "baseline", "basis": BASELINE, "ref": "README §4", "desc": desc,
                    "score": s["score"], "macro_f1": s["macro_f1"], "micro_f1": s["micro_f1"],
                    **{f"f1_{a}": s["per_event"][a] for a in L.ACTS}})
    return out


# ============================================================================ LightGBM（手工 COUNT 特徵）
WINDOWS = (7, 28, 91, 365)


def count_features(li, users: np.ndarray, seed, ctx_vec: np.ndarray, n_users_total: int) -> np.ndarray:
    """每位使用者：6 個活動 × 4 個窗（7/28/91/365 天）的次數 + 距上次的天數（log），全部只用 row time < seed 的列；
    再接上 cohort 情境 ctx。這就是 L5-1 p30 的「hand-engineered features」。"""
    st = np.datetime64(pd.Timestamp(seed))
    cols = []
    for a in L.ACTS:
        u, et, rt = li.src[a]
        hi = np.searchsorted(et, st)
        for w in WINDOWS:
            lo = np.searchsorted(et, st - np.timedelta64(w, "D"))
            m = rt[lo:hi] < st
            cols.append(np.log1p(np.bincount(u[lo:hi][m], minlength=n_users_total)[users]))
        m = rt[:hi] < st
        last = np.full(n_users_total, np.nan)
        uu, ee = u[:hi][m], et[:hi][m]
        if len(uu):
            secs = (st - ee).astype("timedelta64[s]").astype("float64")
            best = np.full(n_users_total, np.inf); np.minimum.at(best, uu, secs)
            last = np.where(np.isinf(best), np.nan, best)
        cols.append(np.log1p(last[users] / 86400.0))
    X = np.stack(cols, 1)
    return np.concatenate([X, np.repeat(ctx_vec[None, :], len(users), 0)], 1)


def run_lgbm(spec: ExpSpec, env) -> dict:
    import lightgbm as lgb
    cfg, plan, li, cohort, eval_u = env["cfg"], env["plan"], env["li"], env["cohort"], env["eval_u"]
    n_tot = env["n_users_total"]
    def table(users, cutoffs, labels=True):
        Xs, Ys = [], []
        for c in cutoffs:
            sd = plan.seed(c)
            Xs.append(count_features(li, users, sd, li.global_ctx(cohort, sd), n_tot))
            Ys.append(li.labels(users, c, c + L.WEEK))
        return np.concatenate(Xs), np.concatenate(Ys)
    t0 = time.time()
    Xtr, Ytr = table(env["train_users"], plan.train_cutoffs)
    Xca, Yca = table(eval_u, [plan.calib_cutoff]); Xva, Yva = table(eval_u, [plan.val_cutoff])
    pc, pv = np.zeros_like(Yca), np.zeros_like(Yva)
    n_est = 30 if cfg.smoke else 400
    for j in range(len(L.ACTS)):
        m = lgb.LGBMClassifier(n_estimators=n_est, learning_rate=0.05, num_leaves=63, subsample=0.8, subsample_freq=1,
                               colsample_bytree=0.8, random_state=spec.seed, n_jobs=os.cpu_count(), verbose=-1)
        m.fit(Xtr, Ytr[:, j])
        pc[:, j] = m.predict_proba(Xca)[:, 1]; pv[:, j] = m.predict_proba(Xva)[:, 1]
    row = eval_block(pc, Yca, pv, Yva)
    row.update(n_train=len(Xtr), epochs_run=0, best_epoch=0, train_sec=time.time() - t0)
    return row


# ============================================================================ GNN 訓練（一個設定）
def run_gnn(spec: ExpSpec, env) -> dict:
    cfg: L.Config = env["cfg"]; plan = env["plan"]; li = env["li"]; cohort = env["cohort"]
    data, col_stats = env["graph"]; device = env["device"]; log = env.get("log", print)
    L.set_seed(cfg.seed)
    train_gap = pd.Timedelta(0) if spec.no_gap_sim else None
    tr = L.make_sample_table(env["train_users"], plan.train_cutoffs, plan, li, cohort, seed_gap=train_gap)
    ca = L.make_sample_table(env["eval_u"], [plan.calib_cutoff], plan, li, cohort)
    va = L.make_sample_table(env["eval_u"], [plan.val_cutoff], plan, li, cohort)
    y_cal, y_val = ca.y.numpy(), va.y.numpy()
    dyn_dims = {nt: 2 * len(c) for nt, c in LATE_COLS.items() if nt in data.node_types} if spec.late_dynamic else None
    model = L.RDLModel(data, col_stats, cfg, ctx_dim=tr.ctx.shape[1], gnn_cls=gnn_factory("sage_keep" if (spec.drop_rev and spec.gnn == "sage") else spec.gnn),
                       dyn_dims=dyn_dims).to(device)
    model.ctx_mean.copy_(tr.ctx.mean(0).to(device)); model.ctx_std.copy_(tr.ctx.std(0).clamp_min(1e-3).to(device))
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    mk = lambda st, sh: L.make_loader(data, st, cfg, sh, transform=ExpTransform(
        "Users", st.y, st.ctx, spec.lookback_days, spec.late_dynamic))
    trl, cal, val = mk(tr, True), mk(ca, False), mk(va, False)
    best, best_state, bad, best_ep, hist = -1.0, None, 0, 0, []
    T0 = time.time()
    for ep in range(1, cfg.epochs + 1):
        t0 = time.time()
        loss = L.train_epoch(model, trl, opt, device, None, cfg.max_steps_per_epoch, log_every=0)
        p_cal, _ = L.predict(model, cal, device)
        ap = mean_ap(p_cal, y_cal)          # epoch 選擇只看 calib 週（不看 val 週）
        hist.append({"epoch": ep, "loss": loss, "calib_mean_ap": ap, "sec": time.time() - t0})
        log(f"    [{spec.name}] epoch {ep}: loss {loss:.4f} | calib mean AP {ap:.4f} | {time.time()-t0:.0f}s")
        if ap > best + 1e-6:
            best, bad, best_ep = ap, 0, ep
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        if bad >= cfg.patience:
            break
        spent = (time.time() - T0) / 60
        if ep < cfg.epochs and spent * (ep + 1) / ep > cfg.time_budget_min:
            log(f"    [{spec.name}] per-config time budget {cfg.time_budget_min} min reached after {ep} epochs")
            break
    train_sec = time.time() - T0
    model.load_state_dict(best_state)
    p_cal, _ = L.predict(model, cal, device); p_val, _ = L.predict(model, val, device)
    row = eval_block(p_cal, y_cal, p_val, y_val)
    row.update(n_train=len(tr.nodes), epochs_run=len(hist), best_epoch=best_ep, train_sec=train_sec,
               sec_per_epoch=train_sec / max(len(hist), 1), n_params=sum(p.numel() for p in model.parameters()),
               history=json.dumps(hist))
    for ld in (trl, cal, val):           # 關掉 persistent workers，否則每個設定會留下 2 個閒置行程（各 1.5–2 GB）
        it = getattr(ld, "_iterator", None)
        if it is not None and hasattr(it, "_shutdown_workers"):
            it._shutdown_workers()
    del trl, cal, val, model, opt
    import gc; gc.collect()
    return row


# ============================================================================ 結果檔與續跑
def load_results(path) -> pd.DataFrame:
    return pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()


def append_result(path, row: dict):
    """欄位可能不同（baseline 列沒有訓練欄位）→ 讀舊檔、合併、整個重寫（先寫暫存檔再換名，避免中斷時檔案壞掉）。"""
    df = pd.concat([load_results(path), pd.DataFrame([row])], ignore_index=True)
    lead = [c for c in META_COLS if c in df.columns]
    df = df[lead + [c for c in df.columns if c not in lead]]
    df.to_csv(path + ".tmp", index=False); os.replace(path + ".tmp", path)


def done_names(path) -> set:
    r = load_results(path)
    return set(r.loc[r.get("status", pd.Series(dtype=str)) == "ok", "name"]) if len(r) else set()


# ============================================================================ 執行整個清單
META_COLS = ["name", "status", "group", "basis", "ref", "desc", "kind", "gnn", "seed", "val_offset", "val_cutoff_utc",
             "calib_cutoff_utc", "n_train_cutoffs", "first_train_cutoff_utc", "last_train_cutoff_utc", "device", "smoke",
             "total_sec", "finished_utc", "data_end_utc", "error"]


def spec_cfg(base_cfg: L.Config, spec: ExpSpec) -> L.Config:
    cfg = copy.deepcopy(base_cfg)
    for k, v in spec.cfg.items():
        setattr(cfg, k, tuple(v) if isinstance(v, list) else v)
    cfg.seed = spec.seed
    if cfg.smoke and "num_neighbors" in spec.cfg:      # SMOKE：鄰居數縮小，但層數保持
        cfg.num_neighbors = tuple(max(2, n // 8) for n in cfg.num_neighbors)
    return cfg


class Runner:
    """依 RUN_LIST 逐一執行；每完成一個就寫入 results.csv；已完成（status=ok）的會跳過。"""

    def __init__(self, base: dict, results_path: str, session_budget_h: float, log=print):
        self.b, self.path, self.budget_s, self.log = base, results_path, session_budget_h * 3600, log
        self.graphs = {}
        self.T0 = time.time(); self.ran_any = False

    def graph_for(self, spec: ExpSpec):
        key = spec.graph_key() + str(spec.late_dynamic)
        if key not in self.graphs:
            self.graphs = {k: v for k, v in self.graphs.items() if k == "base"}   # 只留一份變體，省記憶體
            late = self.b["late"] if spec.late_dynamic else None
            self.graphs[key] = derive_graph(self.b["data"], self.b["col_stats"], spec, late)
        return self.graphs[key]

    def ensure_baselines(self, offsets):
        done = done_names(self.path)
        for k in sorted(set(offsets)):
            plan = L.make_time_plan(self.b["cfg"], self.b["end_excl"], k)
            for row in baseline_rows(self.b["li"], self.b["eval_u"], plan):
                nm = row["name"] + ("" if k == 0 else f"@val-{k}w")
                if nm in done:
                    continue
                row.update(name=nm, status="ok", kind="baseline", val_offset=k, val_cutoff_utc=str(plan.val_cutoff),
                           calib_cutoff_utc=str(plan.calib_cutoff), smoke=self.b["cfg"].smoke,
                           finished_utc=str(pd.Timestamp.utcnow()), data_end_utc=str(self.b["end_excl"]))
                append_result(self.path, row)

    def estimate(self, spec: ExpSpec) -> float:
        """下一個設定的預估秒數：用 results.csv 裡同一台機器（device）已完成設定的中位數；沒有就用設定的時間預算。"""
        r = load_results(self.path)
        if len(r) and "total_sec" in r:
            m = r[(r.status == "ok") & (r.kind == spec.kind) & (r.device == str(self.b["device"])) &
                  (r.smoke == self.b["cfg"].smoke)]["total_sec"].dropna()
            if len(m):
                return float(m.median())
        return 60.0 if spec.kind == "lgbm" else self.b["cfg"].time_budget_min * 60 + 120

    def run(self, specs: List[ExpSpec]):
        self.ensure_baselines([s.val_offset for s in specs])
        for i, spec in enumerate(specs, 1):
            if spec.name in done_names(self.path):
                self.log(f"[{i}/{len(specs)}] {spec.name}: already in results.csv → skip"); continue
            used = time.time() - self.T0
            est = self.estimate(spec)
            if used + est > self.budget_s and self.ran_any:   # 每個 session 至少跑一個，避免卡住
                self.log(f"[{i}/{len(specs)}] STOP: session time used {used/3600:.2f} h + estimate {est/3600:.2f} h "
                         f"> SESSION_BUDGET_H. 其餘設定請用下一個 session / version 續跑（見說明）。")
                return False
            cfg = spec_cfg(self.b["cfg"], spec)
            plan = L.make_time_plan(cfg, self.b["end_excl"], spec.val_offset)
            meta = {"name": spec.name, "group": spec.group, "basis": spec.basis, "ref": spec.ref, "desc": spec.desc,
                    "kind": spec.kind, "gnn": spec.gnn if spec.kind == "gnn" else "", "seed": spec.seed,
                    "val_offset": spec.val_offset, "val_cutoff_utc": str(plan.val_cutoff),
                    "calib_cutoff_utc": str(plan.calib_cutoff), "n_train_cutoffs": len(plan.train_cutoffs),
                    "first_train_cutoff_utc": str(plan.train_cutoffs[0]), "last_train_cutoff_utc": str(plan.train_cutoffs[-1]),
                    "device": str(self.b["device"]), "smoke": cfg.smoke, "data_end_utc": str(self.b["end_excl"]),
                    "config": json.dumps({k: (list(v) if isinstance(v, tuple) else v) for k, v in asdict(cfg).items()}),
                    "spec": json.dumps({k: (list(v) if isinstance(v, tuple) else v) for k, v in asdict(spec).items()})}
            self.log(f"[{i}/{len(specs)}] {spec.name} ({spec.group}, {spec.basis}): {spec.desc} | val {plan.val_cutoff:%Y-%m-%d} "
                     f"| {len(plan.train_cutoffs)} train cutoffs | est {est/60:.0f} min")
            t0 = time.time(); self.ran_any = True
            try:
                env = dict(self.b, cfg=cfg, plan=plan)
                if spec.kind == "lgbm":
                    row = run_lgbm(spec, env)
                else:
                    env["graph"] = self.graph_for(spec)
                    row = run_gnn(spec, env)
                row.update(meta, status="ok", error="")
            except Exception as e:   # 一個設定失敗不影響其他設定
                import traceback; traceback.print_exc()
                row = dict(meta, status="error", error=f"{type(e).__name__}: {e}"[:500])
            row["total_sec"] = time.time() - t0; row["finished_utc"] = str(pd.Timestamp.utcnow())
            append_result(self.path, row)
            if row["status"] == "ok":
                self.log(f"    -> Score {100*row['score']:.2f}% (macro {100*row['macro_f1']:.2f}, micro {100*row['micro_f1']:.2f}) "
                         f"| prob-thr {100*row['score_calib_probthr']:.2f} | {row['total_sec']/60:.1f} min")
            if self.b["device"].type == "cuda":
                torch.cuda.empty_cache()
        return True


# ============================================================================ 結果表與圖
def summary_table(res: pd.DataFrame, val_offset=0) -> pd.DataFrame:
    r = res[(res.status == "ok") & (res.val_offset.fillna(0) == val_offset)].copy()
    cols = ["name", "group", "basis", "desc", "score", "macro_f1", "micro_f1"] + [f"f1_{a}" for a in L.ACTS] + \
           ["score_calib_probthr", "score_th05", "score_val_rate_optimistic", "mean_auc", "best_epoch", "epochs_run", "total_sec"]
    r = r[[c for c in cols if c in r.columns]]
    for c in r.columns:
        if c.startswith(("score", "macro", "micro", "f1_")):
            r[c] = (100 * r[c]).round(2)
    if "total_sec" in r:
        r["total_min"] = (r.pop("total_sec") / 60).round(1)
    return r


def seed_stats(res: pd.DataFrame, prefix: str, val_offset=0):
    r = res[(res.status == "ok") & (res.name.str.match(rf"^{prefix}_s\d+$")) & (res.val_offset.fillna(0) == val_offset)]
    return r


def make_plots(res: pd.DataFrame, out_dir: str) -> List[str]:
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    os.makedirs(out_dir, exist_ok=True); files = []
    ok = res[res.status == "ok"].copy(); ok["val_offset"] = ok.val_offset.fillna(0)
    smoke = bool(ok.get("smoke", pd.Series([False])).fillna(False).astype(str).isin(["True", "true", "1"]).any())
    _orig_savefig = plt.Figure.savefig
    def _save(fig, p, **kw):   # SMOKE 結果的圖加上浮水印，避免誤放進投影片
        if smoke:
            fig.text(0.5, 0.5, "SMOKE TEST - numbers are meaningless", ha="center", va="center", fontsize=34,
                     color="red", alpha=0.25, rotation=20, transform=fig.transFigure)
        _orig_savefig(fig, p, **kw)
    main = ok[ok.val_offset == 0]
    # 1) 每個活動的 F1（主結果）
    groups = [("All zeros", main[main.name == "All zeros"]), ("Copy last week", main[main.name == "Copy last week"]),
              ("Copy last week (gap)", main[main.name == "Copy last week (gap)"]),
              ("LightGBM (counts)", main[main.name == "LightGBM_counts"]),
              ("M1 HeteroGraphSAGE", seed_stats(ok, "M1")), ("M2 RelGNN", seed_stats(ok, "M2_RelGNN"))]
    groups = [(n, g) for n, g in groups if len(g)]
    if groups:
        fig, ax = plt.subplots(figsize=(13.33, 6.0)); w = 0.8 / len(groups); x = np.arange(len(L.ACTS) + 1)
        for i, (n, g) in enumerate(groups):
            vals = g[[f"f1_{a}" for a in L.ACTS] + ["score"]].to_numpy() * 100
            mu, sd = vals.mean(0), (vals.std(0, ddof=1) if len(vals) > 1 else np.zeros(vals.shape[1]))
            lab = n + (f" (n={len(vals)} seeds)" if len(vals) > 1 else "")
            ax.bar(x + i * w - 0.4 + w / 2, mu, w, yerr=sd if len(vals) > 1 else None, capsize=2, label=lab)
        ax.set_xticks(x); ax.set_xticklabels(L.ACTS + ["Score"], rotation=15)
        ax.set_ylabel("F1 / Score on validation week (%)")
        vc = main.val_cutoff_utc.dropna().iloc[0] if len(main.val_cutoff_utc.dropna()) else ""
        ax.set_title(f"Per-activity F1, validation week from {str(vc)[:10]} 16:00 UTC (rate rule tuned on previous week)")
        ax.legend(fontsize=9, ncol=2); ax.grid(axis="y", alpha=.3); fig.tight_layout()
        p = os.path.join(out_dir, "fig_per_activity_f1.png"); _save(fig, p, dpi=200); plt.close(fig); files.append(p)
    # 2) 消融：相對 M1_s42 的 Score 差（Q2、Q3 各一張），灰色區 = M1 三個 seed 的 ±1 標準差
    ref = main[main.name == "M1_s42"]
    m1 = seed_stats(ok, "M1")
    if len(ref):
        base = float(ref.score.iloc[0]) * 100
        sd = float(m1.score.std(ddof=1) * 100) if len(m1) > 1 else 0.0
        for grp in ["Q2", "Q3"]:
            g = main[(main.group == grp) & (~main.name.str.match(r"^M[12]_"))] if grp == "Q3" else \
                main[(main.group == grp) & (~main.name.str.match(r"^M1_"))]
            g = g[~g.name.str.match(r"^M2_RelGNN_s4[34]$")]
            if not len(g):
                continue
            d = (g.score * 100 - base).to_numpy(); o = np.argsort(d)
            fig, ax = plt.subplots(figsize=(13.33, max(3.5, 0.45 * len(g) + 1.5)))
            colors = ["#d62728" if v < 0 else "#2ca02c" for v in d[o]]
            labels = [f"{n}  [{b}]" for n, b in zip(g.desc.to_numpy()[o], g.basis.to_numpy()[o])]
            ax.barh(range(len(g)), d[o], color=colors)
            ax.set_yticks(range(len(g))); ax.set_yticklabels(labels, fontsize=9)
            if sd:
                ax.axvspan(-sd, sd, color="grey", alpha=.2, label=f"M1 seed std ±{sd:.2f}")
                ax.legend(loc="lower right")
            ax.axvline(0, color="k", lw=.8)
            ax.set_xlabel(f"Δ Score vs M1 (seed 42, Score {base:.2f}%)  [percentage points]")
            ax.set_title(f"{grp} ablations (validation week, honest calib→val protocol)")
            ax.grid(axis="x", alpha=.3); fig.tight_layout()
            p = os.path.join(out_dir, f"fig_ablation_{grp}.png"); _save(fig, p, dpi=200); plt.close(fig); files.append(p)
    # 3) 決策規則（rate vs 機率門檻 vs 0.5）
    cand = main[main.name.isin(["M1_s42", "M2_RelGNN_s42", "LightGBM_counts"])]
    if len(cand) and "score_calib_probthr" in cand:
        fig, ax = plt.subplots(figsize=(10, 5)); x = np.arange(len(cand)); w = 0.2
        for i, (c, lab) in enumerate([("score", "rate tuned on calib week (ours)"), ("score_calib_probthr", "prob. threshold tuned on calib week"),
                                      ("score_th05", "threshold 0.5"), ("score_val_rate_optimistic", "rate tuned on val week (optimistic)")]):
            ax.bar(x + (i - 1.5) * w, cand[c] * 100, w, label=lab)
        ax.set_xticks(x); ax.set_xticklabels(cand.name); ax.set_ylabel("Score (%)"); ax.legend(fontsize=9)
        ax.set_title("Decision-rule ablation"); ax.grid(axis="y", alpha=.3); fig.tight_layout()
        p = os.path.join(out_dir, "fig_decision_rule.png"); _save(fig, p, dpi=200); plt.close(fig); files.append(p)
    # 4) 多週回測
    bt = ok[ok.name.str.replace(r"@val-\dw$|_val-\dw$", "", regex=True).isin(
        ["M1_s42", "M2_RelGNN_s42", "LightGBM_counts", "Copy last week", "Copy last week (gap)"])].copy()
    if len(bt) and bt.val_offset.nunique() > 1:
        bt["base"] = bt.name.str.replace(r"@val-\dw$|_val-\dw$", "", regex=True)
        fig, ax = plt.subplots(figsize=(10, 5))
        for n, g in bt.groupby("base"):
            g = g.sort_values("val_cutoff_utc"); ax.plot(pd.to_datetime(g.val_cutoff_utc), g.score * 100, "o-", label=n)
        ax.set_ylabel("Score (%)"); ax.set_xlabel("validation week start (UTC)"); ax.legend(); ax.grid(alpha=.3)
        ax.set_title("Multi-week backtest (each point = retrained with only earlier data)"); fig.tight_layout()
        p = os.path.join(out_dir, "fig_backtest.png"); _save(fig, p, dpi=200); plt.close(fig); files.append(p)
    # 5) 每個設定的執行時間
    rt = ok[ok.kind.isin(["gnn", "lgbm"])]
    if len(rt):
        fig, ax = plt.subplots(figsize=(13.33, max(4, 0.3 * len(rt) + 1)))
        ax.barh(rt.name, rt.total_sec / 60); ax.set_xlabel(f"minutes (device: {', '.join(rt.device.unique())})")
        ax.set_title("Runtime per config"); ax.grid(axis="x", alpha=.3); fig.tight_layout()
        p = os.path.join(out_dir, "fig_runtime.png"); _save(fig, p, dpi=150); plt.close(fig); files.append(p)
    return files


In [ ]:
%%writefile relgnn_vendor.py
# -*- coding: utf-8 -*-
"""RelGNN 官方實作（原封不動複製，僅合併成一個檔案並修改 import）。

來源：https://github.com/snap-stanford/RelGNN （commit cffdb8b54627e92c7dd112c1243dde739c90d35b，MIT License, Copyright (c) 2023 RelBench Team）
檔案：examples/atomic_routes.py、examples/relgnn_conv.py、examples/relgnn_hetero_conv.py、examples/relgnn_nn.py（只取 RelGNN 類別）
論文：Chen, Kanatsoulis, Leskovec. RelGNN: Composite Message Passing for Relational Deep Learning. arXiv:2502.06784 (ICML 2025)
講義：Lecture 5 (2) p6–p9（atomic route、composite message passing、bridge / hub 結構）。
RelGNN 沒有 PyPI 套件（pip install relgnn 找不到），所以把需要的模組放在這裡。
"""
# ---------------------------------------------------------------- examples/atomic_routes.py
from collections import defaultdict

def get_atomic_routes(edge_type_list):
    
    src_to_tuples = defaultdict(list)
    for src, rel, dst in edge_type_list:
        if rel.startswith('f2p'):
            if src == dst:
                src = src + '--' + rel
            src_to_tuples[src].append((src, rel, dst))

    atomic_routes_list = []
    get_rev_edge = lambda edge: (edge[2], 'rev_' + edge[1], edge[0])
    for src, tuples in src_to_tuples.items():
        if '--' in src:
            src = src.split('--')[0]
        if len(tuples) == 1:
            _, rel, dst = tuples[0]
            edge = (src, rel, dst)
            atomic_routes_list.append(('dim-dim',) + edge)
            atomic_routes_list.append(('dim-dim',) + get_rev_edge(edge))
        else:
            for _, rel_q, dst_q in tuples:
                for _, rel_v, dst_v in tuples:
                    if rel_q != rel_v:
                        edge_q = (src, rel_q, dst_q)
                        edge_v = (src, rel_v, dst_v)                   
                        atomic_routes_list.append(('dim-fact-dim',) + edge_q + get_rev_edge(edge_v))

    return atomic_routes_list

# ---------------------------------------------------------------- examples/relgnn_conv.py
from torch_geometric.nn.dense.linear import Linear
from torch_geometric.nn.conv import TransformerConv, SAGEConv

class RelGNNConv(TransformerConv):
    def __init__(
        self,
        attn_type,
        in_channels,
        out_channels,
        heads,
        aggr,
        simplified_MP=False,
        bias=True,
        **kwargs,
    ):
        super().__init__(in_channels, out_channels, heads, bias=bias, **kwargs)
        self.attn_type = attn_type
        if attn_type == 'dim-fact-dim':
            self.aggr_conv = SAGEConv(in_channels, out_channels, aggr=aggr)
        self.simplified_MP = simplified_MP
        self.final_proj = Linear(heads * out_channels, out_channels, bias=bias)
        self.final_proj.reset_parameters()

    def forward(
        self,
        x,
        edge_index,
        edge_attr = None,
        return_attention_weights = None,
    ):
        # dim-dim
        if self.attn_type == 'dim-dim':
            if self.simplified_MP and edge_index.shape[1] == 0:
                return None
            out = super().forward(x, edge_index, edge_attr, return_attention_weights)
            return self.final_proj(out)
        
        # dim-fact-dim
        edge_attn, edge_aggr = edge_index
        
        src_aggr, dst_aggr, dst_attn = x

        if self.simplified_MP:
            if edge_attn.shape[1] == 0:
                return None
            
            if edge_aggr.shape[1] == 0:
                src_attn = dst_aggr
            else:
                src_attn = self.aggr_conv((src_aggr, dst_aggr), edge_aggr)
        else:
            src_attn = self.aggr_conv((src_aggr, dst_aggr), edge_aggr)

        out = super().forward((src_attn, dst_attn), edge_attn, edge_attr, return_attention_weights)

        return self.final_proj(out), src_attn
# ---------------------------------------------------------------- examples/relgnn_hetero_conv.py
import warnings
from typing import Dict, List, Optional

import torch
from torch import Tensor

from torch_geometric.nn.module_dict import ModuleDict
from torch_geometric.typing import NodeType
from torch_geometric.utils.hetero import check_add_self_loops


def group(xs: List[Tensor], aggr: Optional[str]) -> Optional[Tensor]:
    if len(xs) == 0:
        return None
    elif aggr is None:
        return torch.stack(xs, dim=1)
    elif len(xs) == 1:
        return xs[0]
    elif aggr == "cat":
        return torch.cat(xs, dim=-1)
    else:
        out = torch.stack(xs, dim=0)
        out = getattr(torch, aggr)(out, dim=0)
        out = out[0] if isinstance(out, tuple) else out
        return out


class RelGNN_HeteroConv(torch.nn.Module):
    r"""A generic wrapper for computing graph convolution on heterogeneous
    graphs.
    This layer will pass messages from source nodes to target nodes based on
    the bipartite GNN layer given for a specific edge type.
    If multiple relations point to the same destination, their results will be
    aggregated according to :attr:`aggr`.
    In comparison to :meth:`torch_geometric.nn.to_hetero`, this layer is
    especially useful if you want to apply different message passing modules
    for different edge types.

    .. code-block:: python

        hetero_conv = HeteroConv({
            ('paper', 'cites', 'paper'): GCNConv(-1, 64),
            ('author', 'writes', 'paper'): SAGEConv((-1, -1), 64),
            ('paper', 'written_by', 'author'): GATConv((-1, -1), 64),
        }, aggr='sum')

        out_dict = hetero_conv(x_dict, edge_index_dict)

        print(list(out_dict.keys()))
        >>> ['paper', 'author']

    Args:
        convs (Dict[Tuple[str, str, str], MessagePassing]): A dictionary
            holding a bipartite
            :class:`~torch_geometric.nn.conv.MessagePassing` layer for each
            individual edge type.
        aggr (str, optional): The aggregation scheme to use for grouping node
            embeddings generated by different relations
            (:obj:`"sum"`, :obj:`"mean"`, :obj:`"min"`, :obj:`"max"`,
            :obj:`"cat"`, :obj:`None`). (default: :obj:`"sum"`)
    """
    def __init__(
        self,
        convs,
        aggr: Optional[str] = "sum",
        simplified_MP: Optional[bool] = False,
    ):
        super().__init__()

        for edge_type, module in convs.items():
            check_add_self_loops(module, [edge_type])

        src_node_types = {key[0] for key in convs.keys()}
        dst_node_types = {key[-1] for key in convs.keys()}
        if len(src_node_types - dst_node_types) > 0:
            warnings.warn(
                f"There exist node types ({src_node_types - dst_node_types}) "
                f"whose representations do not get updated during message "
                f"passing as they do not occur as destination type in any "
                f"edge type. This may lead to unexpected behavior.")

        self.convs = ModuleDict(convs)
        self.aggr = aggr
        self.simplified_MP = simplified_MP

    def reset_parameters(self):
        r"""Resets all learnable parameters of the module."""
        for conv in self.convs.values():
            conv.reset_parameters()

    def forward(
        self,
        x_dict, 
        edge_index_dict,
    ) -> Dict[NodeType, Tensor]:
        r"""Runs the forward pass of the module.

        Args:
            x_dict (Dict[str, torch.Tensor]): A dictionary holding node feature
                information for each individual node type.
            edge_index_dict (Dict[Tuple[str, str, str], torch.Tensor]): A
                dictionary holding graph connectivity information for each
                individual edge type, either as a :class:`torch.Tensor` of
                shape :obj:`[2, num_edges]` or a
                :class:`torch_sparse.SparseTensor`.
        """
        out_dict: Dict[str, List[Tensor]] = {}
        def update(out_dict, dst, out):
            if dst not in out_dict:
                out_dict[dst] = [out]
            else:
                out_dict[dst].append(out)

        for edge_type_info, conv in self.convs.items():
            attn_type = edge_type_info[0]

            if attn_type == 'dim-dim':
                src, rel, dst = edge_type_info[1:]
                x = (
                        x_dict.get(src, None),
                        x_dict.get(dst, None),
                    )
                edge_index = edge_index_dict[(src, rel, dst)]

                out = conv(x, edge_index)

                if self.simplified_MP and out is None:
                    continue

                update(out_dict, dst, out)
                        
            elif attn_type == 'dim-fact-dim':
                edge_attn, edge_aggr = edge_type_info[1:4], edge_type_info[4:]
                src_attn, _, dst = edge_attn
                src_aggr = edge_aggr[0]
                x = (
                        x_dict[src_aggr],
                        x_dict[src_attn],
                        x_dict[dst],
                    )
                edge_index = (
                        edge_index_dict[edge_attn],
                        edge_index_dict[edge_aggr],
                    )
                out = conv(x, edge_index)
                
                if self.simplified_MP and out is None:
                    continue

                out_dst, out_src_attn = out
                update(out_dict, dst, out_dst)
                update(out_dict, src_attn, out_src_attn)            


        for key, value in out_dict.items():
            out_dict[key] = group(value, self.aggr)

        if self.simplified_MP:
            for key, value in x_dict.items():
                if key not in out_dict:
                    out_dict[key] = value

        return out_dict

    def __repr__(self) -> str:
        return f'{self.__class__.__name__}(num_relations={len(self.convs)})'

# ---------------------------------------------------------------- examples/relgnn_nn.py（RelGNN 類別）
from typing import Any, Dict, List, Optional
from torch import Tensor
from torch_geometric.nn import LayerNorm
from torch_geometric.typing import EdgeType, NodeType
class RelGNN(torch.nn.Module):
    def __init__(
        self,
        node_types: List[NodeType],
        edge_types: List[EdgeType],
        channels: int,
        aggr: str = "sum",
        num_model_layers: int = 2,
        num_heads: int = 1,
        simplified_MP=False,
    ):
        super().__init__()

        self.convs = torch.nn.ModuleList()
        for _ in range(num_model_layers):
            conv = RelGNN_HeteroConv(
                {
                    edge_type: RelGNNConv(edge_type[0], (channels, channels), channels, num_heads, aggr=aggr, simplified_MP=simplified_MP)
                    for edge_type in edge_types
                },
                aggr=aggr,
                simplified_MP=simplified_MP,
            )
            self.convs.append(conv)

        self.norms = torch.nn.ModuleList()
        for _ in range(num_model_layers):
            norm_dict = torch.nn.ModuleDict()
            for node_type in node_types:
                norm_dict[node_type] = LayerNorm(channels, mode="node")
            self.norms.append(norm_dict)

    def reset_parameters(self):
        for conv in self.convs:
            conv.reset_parameters()
        for norm_dict in self.norms:
            for norm in norm_dict.values():
                norm.reset_parameters()

    def forward(
        self,
        x_dict: Dict[NodeType, Tensor],
        edge_index_dict: Dict[NodeType, Tensor],
        num_sampled_nodes_dict: Optional[Dict[NodeType, List[int]]] = None,
        num_sampled_edges_dict: Optional[Dict[EdgeType, List[int]]] = None,
    ) -> Dict[NodeType, Tensor]:
        for _, (conv, norm_dict) in enumerate(zip(self.convs, self.norms)):
            x_dict = conv(x_dict, edge_index_dict)
            x_dict = {key: norm_dict[key](x) for key, x in x_dict.items()}
            x_dict = {key: x.relu() for key, x in x_dict.items()}

        return x_dict


In [ ]:
import sys, time, shutil, glob, importlib
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd, torch
import asg2_lib as L, asg2_exp as E, relgnn_vendor
for m in (L, relgnn_vendor, E): importlib.reload(m)

cfg = L.Config(seed=SEED_BASE, smoke=SMOKE)
cfg.time_budget_min = float(PER_CONFIG_MAX_MIN)
if SMOKE:   # 只為了確認每個設定的程式路徑都能跑；數字沒有意義
    cfg.max_train_cutoffs, cfg.epochs, cfg.patience = 2, 1, 1
    cfg.max_steps_per_epoch, cfg.channels, cfg.batch_size = 8, 32, 256
    cfg.num_neighbors, cfg.num_workers = (16, 8), 0
for k, v in CFG_OVERRIDE.items():
    setattr(cfg, k, tuple(v) if isinstance(v, list) else v)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(os.cpu_count() or 1)
VERSIONS = L.versions(); print(json.dumps(VERSIONS, indent=1)); print("device:", DEVICE); print(cfg)

## 4. 讀資料、建完整的圖（每個設定再從這張圖拿掉表/邊/欄位，不重建）

In [ ]:
t0 = time.time()
DATA_DIR = L.find_data_dir(DATA_ROOT); MANIFEST = L.find_manifest(DATA_DIR, DATA_ROOT)
feat, stypes, raw = L.load_tables(DATA_DIR, cfg)
END_EXCL, _ = L.data_end_exclusive(MANIFEST, feat)
manifest = json.load(open(MANIFEST)) if MANIFEST else {}
print("data:", DATA_DIR, "| period (UTC):", manifest.get("start_utc"), "..", manifest.get("end_utc_inclusive"))
print(L.make_time_plan(cfg, END_EXCL).describe())
data, col_stats = L.build_graph(feat, stypes)
late = E.load_late(DATA_DIR)
li = L.LabelIndex(raw); users = raw["Users"]
cohort = users.user_id[users.is_cohort.astype(bool)].to_numpy()
test_df = pd.read_csv(os.path.join(REPO_DIR, "submission", "test_input.csv"))
eval_u = users.set_index("github_user_id").user_id.loc[test_df.UserID].to_numpy()
train_users = cohort if not SMOKE else np.random.RandomState(0).choice(cohort, 600, replace=False)
del feat
print(f"graph: {len(data.node_types)} node types, {len(data.edge_types)} edge types | "
      f"train users {len(train_users):,} | eval users {len(eval_u):,} | {time.time()-t0:.0f}s")

## 5. 實驗清單與預估時間
下表列出這次要跑的設定（依優先順序）。**預估時間**：一開始沒有 GPU 實測，只能用「每個設定的時間預算」當上限；
跑完第一個設定後，之後的預估會改用 `results.csv` 裡同一台機器的實際中位數。box（8 核 CPU）上的實測見 `PROGRESS_exp.md` /
交付報告（**那是 CPU 數字，不是 GPU**）。

In [ ]:
SPECS = E.default_specs()
by_name = {s.name: s for s in SPECS}
run_specs = SPECS if RUN_LIST == "ALL" else [by_name[n] for n in RUN_LIST]
# 續跑：如果 Input 裡有之前 version 的 results.csv，合併進來（已完成的設定會跳過）
prev = [p for p in glob.glob(os.path.join(DATA_ROOT, "**", os.path.basename(RESULTS)), recursive=True)
        if os.path.abspath(p) != os.path.abspath(RESULTS)]
if prev:
    old = pd.concat([pd.read_csv(p) for p in prev], ignore_index=True)
    cur = E.load_results(RESULTS)
    allr = pd.concat([cur, old], ignore_index=True).drop_duplicates(subset=["name", "status"], keep="first")
    allr.to_csv(RESULTS, index=False); print("merged previous results from", prev, "->", len(allr), "rows")
done = E.done_names(RESULTS)
todo = [s for s in run_specs if s.name not in done]
r0 = E.load_results(RESULTS)
n_g, n_l = sum(1 for s in todo if s.kind == "gnn"), sum(1 for s in todo if s.kind == "lgbm")
print(f"{len(run_specs)} configs in RUN_LIST, {len(run_specs)-len(todo)} already done, {len(todo)} to run ({n_g} GNN, {n_l} LightGBM)")
med = {}
if len(r0) and "total_sec" in r0:
    ok0 = r0[(r0.status == "ok") & (r0.device == str(DEVICE)) & (r0.smoke.astype(str) == str(SMOKE))]
    med = ok0.groupby("kind").total_sec.median().to_dict()
if "gnn" in med:
    est_h = (n_g * med["gnn"] + n_l * med.get("lgbm", 300)) / 3600
    print(f"estimate from finished configs on this device ({DEVICE}): GNN median {med['gnn']/60:.0f} min "
          f"-> remaining ≈ {est_h:.1f} h ≈ {int(np.ceil(est_h / SESSION_BUDGET_H))} session(s) of {SESSION_BUDGET_H} h")
else:
    print("no finished GNN config on this device yet -> the estimate appears after the first config "
          "(each GNN config: up to cfg.epochs epochs with early stopping, hard cap PER_CONFIG_MAX_MIN = "
          f"{PER_CONFIG_MAX_MIN} min). Box CPU measurements are in EXP_GUIDE_zh.md (CPU, not GPU).")
print(f"this session stops before a config that would end after SESSION_BUDGET_H = {SESSION_BUDGET_H} h")
pd.DataFrame([{"name": s.name, "group": s.group, "basis": s.basis, "ref": s.ref, "desc": s.desc,
               "done": s.name in done} for s in run_specs])

## 6. 執行（每完成一個設定就寫入 results.csv）

In [ ]:
BASE = dict(cfg=cfg, end_excl=END_EXCL, li=li, cohort=cohort, eval_u=eval_u, train_users=train_users,
            data=data, col_stats=col_stats, late=late, device=DEVICE, n_users_total=len(users))
runner = E.Runner(BASE, RESULTS, SESSION_BUDGET_H)
finished = runner.run(run_specs)
print("ALL DONE" if finished else "stopped early (session budget) — continue in the next session")

## 7. 結果表（val 週；Score = (macro-F1 + micro-F1)/2，%）

In [ ]:
res = E.load_results(RESULTS)
pd.set_option("display.max_rows", 200, "display.width", 250)
tab = E.summary_table(res, 0)
tab.to_csv(os.path.join(WORK_DIR, "results_main_week.csv"), index=False)
display(tab) if "display" in dir() else print(tab.to_string())
for pre in ["M1", "M2_RelGNN"]:
    g = E.seed_stats(res, pre)
    if len(g):
        print(f"{pre}: {len(g)} seeds, Score mean {100*g.score.mean():.2f}% ± {100*g.score.std(ddof=1) if len(g)>1 else 0:.2f} (std)")
bt = res[(res.status == "ok") & (res.val_offset.fillna(0) > 0)]
if len(bt):
    print("\nbacktest weeks:"); print(bt[["name", "val_cutoff_utc", "score"]].assign(score=lambda d: (100*d.score).round(2)).to_string(index=False))

## 8. 投影片用的圖（PNG，存在 `/kaggle/working/figs/`）

In [ ]:
files = E.make_plots(res, os.path.join(WORK_DIR, "figs"))
print("\n".join(files))
try:
    from IPython.display import Image, display as _d
    for f in files: _d(Image(filename=f, width=900))
except Exception as e:
    print(e)

## 9. 分多個 session 續跑（完整清單在 GPU 上可能超過 12 小時）
1. 用 **Save Version → Save & Run All (Commit)** 背景執行。這個 session 會在 `SESSION_BUDGET_H` 前自動停下，並把 `results.csv`、圖存到 Output。
2. 下一次：打開同一個 notebook → 右側 **Add Input → Your Work（Notebook Output）** → 選這個 notebook 上一個 version 的 output。
3. 再 **Save Version**。第 5 節會自動找到 `/kaggle/input/**/results.csv` 合併，已完成的設定會跳過，只跑剩下的。
4. 重複直到第 6 節印出 `ALL DONE`。最後一個 version 的 `results.csv` 與 `figs/` 就是完整結果。

**注意**：`SMOKE=True` 的結果寫在 `results_smoke.csv`，不會跟正式結果混在一起。